# CS3244 Group 17 Project: IMDb Spoiler Detection from Movie Reviews

This notebook focuses on the model-training stage for binary spoiler classification on IMDb reviews. It loads preprocessed artifact splits and precomputed feature sets to tune, train, and evaluate classifiers.

Workflow: setup and artifact loading, model training and hyperparameter tuning, model evaluation.

## 1. Setup and Data Loading

In [1]:
# imports
import pandas as pd
import numpy as np
import joblib, pickle, re, html, unicodedata
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from pathlib import Path
from scipy import sparse
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, ParameterSampler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, fbeta_score, precision_score, recall_score, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm

In [3]:
# # Read review dataset
# df = pd.read_json("IMDB_reviews.json", lines=True)

## 2. Data Cleaning and Preprocessing
This section performs cleaning and preprocessing to build the master dataset used across all downstream modeling steps.

In [4]:
# # create master dataset
# df_master = df.copy()

In [5]:
# # Remove duplicate reviews which have different label assigned to the same text
# conflicts = df_master.groupby("review_text")["is_spoiler"].nunique()
# conflicts = conflicts[conflicts > 1].index

# df_master = df_master[~df_master["review_text"].isin(conflicts)]

# # Remove exact duplicate reviews to avoid redundancy and data leakage
# df_master = df_master.drop_duplicates(subset=["review_text"])

# # Remove extremely short reviews that are likely low-information or noisy
# df_master = df_master[df_master["review_text"].str.len() > 25]

# # Remove extremely long reviews that may distort feature distributions and increase computational cost
# df_master = df_master[df_master["review_text"].str.len() < 10000]

# # Add review length as a feature due to strong correlation with spoiler labels
# df_master["length"] = df_master["review_text"].str.len()

# # Reset index after row removals to keep the cleaned dataset tidy and consistent
# df_master = df_master.reset_index(drop=True)

In [6]:
# # Review date stored as object, convert to datetime
# df_master["review_date"] = pd.to_datetime(df_master["review_date"])

In [7]:
# # Basic data cleaning function
# def basic_clean(text):
#     text = str(text) # Forces text to be a String
#     text = html.unescape(text) # Converts HTML to normal text
#     text = unicodedata.normalize("NFKC", text) # Normalizes unicode into a consistent canonical form
#     text = re.sub(r"<[^>]+>", " ", text) # Removes HTML tags 
#     text = text.lower() # Converts all text to lowercase
#     text = re.sub(r"\s+", " ", text) # Collapses all whitespace into a single space
#     text = text.strip() # Removes whitespace in front and behind of text
#     return text

In [8]:
# # Clean "clean_text" in master dataframe
# df_master["clean_text"] = df_master["review_text"].apply(basic_clean)

## 3. Data Splitting
This notebook reuses the grouped movie-level train/test split and the artifact-specific train, validation, and fulltrain datasets created in the preprocessing notebook.
The commented cells below are kept only as reference for how those splits were originally constructed.

In [9]:
# # Function for splitting data into training and testing sets
# def group_data_split(df, group_col="movie_id", test_size=0.2, random_state=42):
#     """
#     Split a dataframe into train and test sets using group-based splitting.

#     Parameters
#     ----------
#     df : pandas.DataFrame
#         Input dataframe.
#     group_col : str, default="movie_id"
#         Column used to define groups that must not be split across train and test.
#     test_size : float, default=0.2
#         Proportion of groups to place in the test set.
#     random_state : int, default=42
#         Random seed for reproducibility.

#     Returns
#     -------
#     df_train : pandas.DataFrame
#         Training subset.
#     df_test : pandas.DataFrame
#         Test subset.
#     """
#     gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)

#     train_idx, test_idx = next(gss.split(df, groups=df[group_col]))

#     df_train = df.iloc[train_idx].reset_index(drop=True)
#     df_test = df.iloc[test_idx].reset_index(drop=True)

#     return df_train, df_test

In [10]:
# # Splitting data into training set and testing set (grouped by movie_id)
# df_train, df_test = group_data_split(
#     df_master, 
#     group_col="movie_id", 
#     test_size=0.2, 
#     random_state=42
# )

In [11]:
# # Create Stratified Group K-Fold splits on the training set
# N_SPLITS = 5
# CV_SEED = 42
# cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=CV_SEED)

# cv_splits = list(
#     cv.split(
#         df_train["clean_text"],
#         df_train["is_spoiler"],
#         groups=df_train["movie_id"],
#     )
# )
# print("Number of CV folds:", len(cv_splits))

## 4. Model Training and Hyperparameter Tuning

This section uses a fixed artifact-based model-selection protocol to keep comparisons fair and avoid leakage.

- Each artifact stream provides a train split for fitting and a validation split for hyperparameter selection.
- Selected hyperparameters are then retrained on the corresponding fulltrain split.
- The held-out df_test split is reserved for final evaluation in Section 5.

For each model family, the workflow is:
1. Train on the artifact-specific train split (done in preprocessing)
2. Tune hyperparameters on the validation split.
3. Refit on the corresponding fulltrain split.
4. Evaluate once on the held-out test split in Section 5.

### 4.1 Shared Tuning Settings

All models in Section 4 select hyperparameters primarily by validation AUC using a shared tuning setup. The search ranges are defined once below for consistency, while feature-specific implementation details such as sparse versus dense preprocessing remain inside the individual model cells.

This subsection contains the shared scoring metric, random seed, and search ranges used during model selection. The fast-rerun fixed hyperparameters are separated into the next subsection.

In [2]:
SCORING_METRIC = "roc_auc"
RANDOM_STATE = 42
CLASS_WEIGHT = "balanced"

# Logistic Regression
LR_C_GRID = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
SBERT_LR_C_GRID = [0.001, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
ROBERTA_LR_C_GRID = [0.001, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
LR_MAX_ITER = 2000

# SVM
SVM_C_GRID = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
SVM_TOL = 1e-4
SVM_MAX_ITER = 3000
SVM_CALIBRATION_CV = 3
SVM_CALIBRATION_METHOD = "sigmoid"

# Tree-based models
DT_PARAM_GRID = {
    "max_depth": [5, 10, 15],
    "min_samples_split": [10, 25, 50],
    "min_samples_leaf": [5, 10, 20],
    "ccp_alpha": [0.0, 1e-4, 1e-3, 1e-2],
}
DT_RANDOM_SEARCH_ITER = 10

RF_PARAM_GRID = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt"]
}
RF_RANDOM_SEARCH_ITER = 5

XGB_PARAM_GRID = {
    "n_estimators": [100, 200, 400],
    "max_depth": [3, 4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1.0, 3.0, 5.0],
}
XGB_RANDOM_SEARCH_ITER = 10

# Naive Bayes
NB_ALPHA_GRID = [0.005, 0.01, 0.05, 0.1, 0.3, 0.5, 1.0]
NB_THRESHOLD_GRID = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

### 4.2 Fast Rerun Settings

This subsection stores fixed hyperparameters from earlier results so repeated runs can skip tuning and go straight to refitting on the full training split.

Set `USE_FIXED_BEST_PARAMS = True` to use the saved values below when they are available. Leave a value as `None` if that model has not completed its first tuning run yet.

We use a two-item list to indicate the optimal paramters for each artifact, where the first entry is for `artifacts_1` and the second try is for `artifacts_2`.

In [3]:
USE_FIXED_BEST_PARAMS = True

# Logistic Regression
LR_TFIDF_FIXED_C = [0.3, 0.3]
LR_TFIDF_LEMMA_FIXED_C = [0.3, 0.3]
LR_SBERT_FIXED_C = [0.3, 10.0]
LR_ROBERTA_FIXED_C = [0.001, None]  # e.g. [0.001, 0.003]

# SVM
SVM_TFIDF_FIXED_C = [0.1, 0.1]
SVM_SBERT_FIXED_C = [None, None]  # e.g. [3.0, 1.0]

# Tree-based models
DT_TFIDF_FIXED_PARAMS = {'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 5, 'ccp_alpha': 0.0001}
DT_SBERT_FIXED_PARAMS = None # {'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 5, 'ccp_alpha': 0.0001}
RF_TFIDF_FIXED_PARAMS = {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2}
RF_SBERT_FIXED_PARAMS = None # {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2}
XGB_TFIDF_FIXED_PARAMS = [None, None]  # e.g. [{'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.85, 'colsample_bytree': 0.8, 'min_child_weight': 1.0}, {...}]
XGB_SBERT_FIXED_PARAMS = [None, None]  # e.g. [{'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.03, 'subsample': 0.85, 'colsample_bytree': 0.8, 'min_child_weight': 3.0}, {...}]

# Naive Bayes
NB_TFIDF_FIXED_ALPHA = None
NB_TFIDF_FIXED_THRESHOLD = None
NB_SBERT_FIXED_ALPHA = None
NB_SBERT_FIXED_THRESHOLD = None

### 4.3 Feature Artifact Loading

Run the next cell to load the two active artifact streams (`artifacts_1` and `artifacts_2`) into memory.
TF-IDF and TF-IDF + lemmatization artifacts stay at the top level of each artifact directory.
SBERT and RoBERTa artifacts are stored by split inside `train_sub`, `train_full`, `val`, and `test`. Each split folder contains its own `bert_window_data.pkl` together with the corresponding `X_sbert.npy` and `X_roberta.npy` files.

Downstream modeling follows a single workflow after loading:
- use the training split for fitting candidate models
- use the validation split for model selection
- refit the chosen model on the full training split
- keep the refit model for held-out test evaluation in section 5

In [4]:
# Load the two active artifact streams used downstream.
ARTIFACT_DIRS = {
    "artifacts_1": Path("artifacts_1"),
    "artifacts_2": Path("artifacts_2"),
}

def _require(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required artifact file: {path}")
    return path

def _load_array(path: Path):
    return np.load(_require(path))

def _load_sparse_matrix(path: Path):
    return sparse.load_npz(_require(path))

def _load_optional_joblib(path: Path):
    return joblib.load(path) if path.exists() else None

def load_artifact_bundle(artifact_name: str):
    if artifact_name not in ARTIFACT_DIRS:
        raise ValueError(f"Unknown artifact_name={artifact_name}. Expected one of {list(ARTIFACT_DIRS)}")

    artifact_dir = ARTIFACT_DIRS[artifact_name]
    if not artifact_dir.exists():
        raise FileNotFoundError(f"Artifact directory not found: {artifact_dir}")

    return {
        "artifact_dir": artifact_dir,
        
        "labels_train": _load_array(artifact_dir / "labels_train.npy"),
        "labels_val": _load_array(artifact_dir / "labels_val.npy"),
        "labels_test": _load_array(artifact_dir / "labels_test.npy"),
        "labels_fulltrain": _load_array(artifact_dir / "labels_fulltrain.npy"),

        "X_tfidf_train": _load_sparse_matrix(artifact_dir / "X_tfidf_train.npz"),
        "X_tfidf_val": _load_sparse_matrix(artifact_dir / "X_tfidf_val.npz"),
        "X_tfidf_test": _load_sparse_matrix(artifact_dir / "X_tfidf_test.npz"),
        "X_tfidf_fulltrain": _load_sparse_matrix(artifact_dir / "X_tfidf_fulltrain.npz"),
        "X_tfidf_test_fulltrain": _load_sparse_matrix(artifact_dir / "X_tfidf_test_fulltrain.npz"),

        "X_tfidf_lemma_train": _load_sparse_matrix(artifact_dir / "X_tfidf_lemma_train.npz"),
        "X_tfidf_lemma_val": _load_sparse_matrix(artifact_dir / "X_tfidf_lemma_val.npz"),
        "X_tfidf_lemma_test": _load_sparse_matrix(artifact_dir / "X_tfidf_lemma_test.npz"),
        "X_tfidf_lemma_fulltrain": _load_sparse_matrix(artifact_dir / "X_tfidf_lemma_fulltrain.npz"),
        "X_tfidf_lemma_test_fulltrain": _load_sparse_matrix(artifact_dir / "X_tfidf_lemma_test_fulltrain.npz"),

        "tfidf_vectorizer": _load_optional_joblib(artifact_dir / "tfidf_vectorizer.joblib"),
        "tfidf_vectorizer_fulltrain": _load_optional_joblib(artifact_dir / "tfidf_vectorizer_fulltrain.joblib"),
        "tfidf_lemma_vectorizer": _load_optional_joblib(artifact_dir / "tfidf_lemma_vectorizer.joblib"),
        "tfidf_lemma_vectorizer_fulltrain": _load_optional_joblib(artifact_dir / "tfidf_lemma_vectorizer_fulltrain.joblib"),
    }

def load_all_artifacts():
    return {name: load_artifact_bundle(name) for name in ARTIFACT_DIRS}

ARTIFACT_STORE = load_all_artifacts()

def load_validation_split(artifact_name="artifacts_1"):
    b = ARTIFACT_STORE[artifact_name]
    return {
        "y_train": b["labels_train"],
        "y_val": b["labels_val"],
        "X_tfidf_train": b["X_tfidf_train"],
        "X_tfidf_val": b["X_tfidf_val"],
        "X_tfidf_lemma_train": b["X_tfidf_lemma_train"],
        "X_tfidf_lemma_val": b["X_tfidf_lemma_val"],
    }

def load_full_train_test(artifact_name="artifacts_1", use_fulltrain=True):
    b = ARTIFACT_STORE[artifact_name]
    if use_fulltrain:
        return {
            "y_train_full": b["labels_fulltrain"],
            "y_test": b["labels_test"],
            "X_tfidf_train_full": b["X_tfidf_fulltrain"],
            "X_tfidf_test": b["X_tfidf_test_fulltrain"],
            "X_tfidf_lemma_train_full": b["X_tfidf_lemma_fulltrain"],
            "X_tfidf_lemma_test": b["X_tfidf_lemma_test_fulltrain"],
        }

    return {
        "y_train_full": b["labels_train"],
        "y_test": b["labels_test"],
        "X_tfidf_train_full": b["X_tfidf_train"],
        "X_tfidf_test": b["X_tfidf_test"],
        "X_tfidf_lemma_train_full": b["X_tfidf_lemma_train"],
        "X_tfidf_lemma_test": b["X_tfidf_lemma_test"],
    }

def load_transformer_artifact_split(artifact_name: str, split_name: str, model_type: str):
    artifact_dir = ARTIFACT_DIRS[artifact_name]
    split_dir = artifact_dir / split_name

    # Each split folder stores its own metadata together with the SBERT and RoBERTa arrays.
    with open(_require(split_dir / "bert_window_data.pkl"), "rb") as f:
        bert_window_data = pickle.load(f)

    if model_type == "sbert":
        X = np.load(_require(split_dir / "X_sbert.npy"))
    elif model_type == "roberta":
        X = np.load(_require(split_dir / "X_roberta.npy"))
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    return {
        "X": X,
        "labels": np.asarray(bert_window_data["labels"], dtype=np.int64),
        "review_ids": np.asarray(bert_window_data["review_ids"]),
        "review_label_map": bert_window_data["review_label_map"],
        "bert_window_data": bert_window_data,
    }

def load_transformer_validation_splits(artifact_name="artifacts_1", model_type="sbert"):
    train = load_transformer_artifact_split(artifact_name, "train_sub", model_type)
    val = load_transformer_artifact_split(artifact_name, "val", model_type)

    return {
        "X_train": train["X"],
        "y_train": train["labels"],
        "review_ids_train": train["review_ids"],
        "review_label_map_train": train["review_label_map"],
        "X_val": val["X"],
        "y_val": val["labels"],
        "review_ids_val": val["review_ids"],
        "review_label_map_val": val["review_label_map"],
    }

def load_transformer_fulltrain_test_splits(artifact_name="artifacts_1", model_type="sbert"):
    train_full = load_transformer_artifact_split(artifact_name, "train_full", model_type)
    test = load_transformer_artifact_split(artifact_name, "test", model_type)

    return {
        "X_train_full": train_full["X"],
        "y_train_full": train_full["labels"],
        "review_ids_train_full": train_full["review_ids"],
        "review_label_map_train_full": train_full["review_label_map"],
        "X_test": test["X"],
        "y_test": test["labels"],
        "review_ids_test": test["review_ids"],
        "review_label_map_test": test["review_label_map"],
    }

def load_transformer_artifact_split_windows(artifact_name: str, split_name: str, model_type=None):
    artifact_dir = ARTIFACT_DIRS[artifact_name]
    with open(_require(artifact_dir / split_name / "bert_window_data.pkl"), "rb") as f:
        return pickle.load(f)

### 4.4 Logistic Regression

#### 4.4.1 TF-IDF

In [17]:
from sklearn.linear_model import LogisticRegression

SUBSET_TO_ARTIFACT = {
    "1": "artifacts_1",
    "2": "artifacts_2",
}

metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
logreg_tfidf_best_c_by_subset = {}
logreg_tfidf_models = {}
logreg_tfidf_tuning_results = {}
tfidf_validation_rows = []

for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    print(f"\n===== Logistic Regression TF-IDF | subset {subset_name} ({artifact_name}) =====")

    # Validation split for model selection.
    validation_artifacts = load_validation_split(artifact_name=artifact_name)
    X_train = validation_artifacts["X_tfidf_train"]
    X_val = validation_artifacts["X_tfidf_val"]
    y_train = validation_artifacts["y_train"]
    y_val = validation_artifacts["y_val"]

    if X_train.shape[0] != len(y_train) or X_val.shape[0] != len(y_val):
        raise ValueError(
            f"TF-IDF artifact mismatch for {artifact_name}: "
            f"train={X_train.shape[0]}/{len(y_train)}, val={X_val.shape[0]}/{len(y_val)}"
        )

    artifact_idx = int(subset_name) - 1
    fixed_c = LR_TFIDF_FIXED_C[artifact_idx] if artifact_idx < len(LR_TFIDF_FIXED_C) else None

    if USE_FIXED_BEST_PARAMS and fixed_c is not None:
        candidate_cs = [float(fixed_c)]
        print(f"Using fixed C for subset {subset_name}: {candidate_cs[0]}")
    else:
        candidate_cs = LR_C_GRID

    tuning_rows = []
    for c_value in tqdm(candidate_cs, desc=f"subset {subset_name} C search", leave=False):
        model = LogisticRegression(
            C=float(c_value),
            solver="saga",
            max_iter=LR_MAX_ITER,
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        )
        model.fit(X_train, y_train)

        y_val_prob = model.predict_proba(X_val)[:, 1]
        tuning_rows.append(
            {
                "C": float(c_value),
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            }
        )

    tuning_df = pd.DataFrame(tuning_rows).sort_values("auc", ascending=False).reset_index(drop=True)
    best_c = float(tuning_df.loc[0, "C"] )

    logreg_tfidf_tuning_results[subset_name] = tuning_df
    logreg_tfidf_best_c_by_subset[subset_name] = best_c

    print(f"Selected C for subset {subset_name}: {best_c}")
    display(
        tuning_df[["C"] + metric_cols]
        .rename(columns={"average_precision": "pr-auc"})
        .round(6)
    )

    selected_row = tuning_df.loc[0, ["C"] + metric_cols].to_dict()
    tfidf_validation_rows.append({"subset": subset_name, **selected_row})

    # Refit on the full-train split for final evaluation.
    full_artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    X_train_full = full_artifacts["X_tfidf_train_full"]
    y_train_full = full_artifacts["y_train_full"]

    final_model = LogisticRegression(
        C=best_c,
        solver="saga",
        max_iter=LR_MAX_ITER,
        random_state=RANDOM_STATE,
        class_weight=CLASS_WEIGHT,
    )
    final_model.fit(X_train_full, y_train_full)
    logreg_tfidf_models[subset_name] = final_model

tfidf_validation_metrics_df = pd.DataFrame(tfidf_validation_rows)
print("Validation metrics for selected C by subset:")
display(
    tfidf_validation_metrics_df[["subset", "C"] + metric_cols]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

validation_details_tfidf = tfidf_validation_metrics_df.copy()
validation_summary_tfidf = tfidf_validation_metrics_df[metric_cols].mean().to_frame().T

print("Average validation metrics across subsets:")
display(
    validation_summary_tfidf.rename(columns={"average_precision": "pr-auc"}).round(6)
)

logreg_tfidf_best_c = logreg_tfidf_best_c_by_subset["1"]
logreg_tfidf_model = logreg_tfidf_models["1"]


===== Logistic Regression TF-IDF | subset 1 (artifacts_1) =====
Using fixed C for subset 1: 0.3


subset 1 C search:   0%|          | 0/1 [00:00<?, ?it/s]

Selected C for subset 1: 0.3


,C,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.3,0.774593,0.567331,0.550689,0.614338,0.469599,0.665627,0.719571



===== Logistic Regression TF-IDF | subset 2 (artifacts_2) =====
Using fixed C for subset 2: 0.3


...

,C,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.3,0.774111,0.566724,0.550474,0.613291,0.470205,0.66379,0.720103


Validation metrics for selected C by subset:


,subset,C,auc,pr-auc,f1,f2,precision,recall,accuracy
0,1,0.3,0.774593,0.567331,0.550689,0.614338,0.469599,0.665627,0.719571
1,2,0.3,0.774111,0.566724,0.550474,0.613291,0.470205,0.663790,0.720103


Average validation metrics across subsets:


,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.774352,0.567028,0.550581,0.613815,0.469902,0.664709,0.719837


#### 4.4.2 TF-IDF + Lemmatization

In [18]:
from sklearn.linear_model import LogisticRegression

SUBSET_TO_ARTIFACT = {
    "1": "artifacts_1",
    "2": "artifacts_2",
}

metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
logreg_tfidf_lemma_best_c_by_subset = {}
logreg_tfidf_lemma_models = {}
logreg_tfidf_lemma_tuning_results = {}
tfidf_lemma_validation_rows = []

for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    print(f"\n===== Logistic Regression TF-IDF + Lemma | subset {subset_name} ({artifact_name}) =====")

    # Validation split for model selection.
    validation_artifacts = load_validation_split(artifact_name=artifact_name)
    X_train = validation_artifacts["X_tfidf_lemma_train"]
    X_val = validation_artifacts["X_tfidf_lemma_val"]
    y_train = validation_artifacts["y_train"]
    y_val = validation_artifacts["y_val"]

    if X_train.shape[0] != len(y_train) or X_val.shape[0] != len(y_val):
        raise ValueError(
            f"TF-IDF+Lemma artifact mismatch for {artifact_name}: "
            f"train={X_train.shape[0]}/{len(y_train)}, val={X_val.shape[0]}/{len(y_val)}. "
            "Regenerate the lemma artifacts before rerunning Logistic Regression 4.4.2."
        )

    artifact_idx = int(subset_name) - 1
    fixed_c = LR_TFIDF_LEMMA_FIXED_C[artifact_idx] if artifact_idx < len(LR_TFIDF_LEMMA_FIXED_C) else None

    if USE_FIXED_BEST_PARAMS and fixed_c is not None:
        candidate_cs = [float(fixed_c)]
        print(f"Using fixed C for subset {subset_name}: {candidate_cs[0]}")
    else:
        candidate_cs = LR_C_GRID

    tuning_rows = []
    for c_value in tqdm(candidate_cs, desc=f"subset {subset_name} C search", leave=False):
        model = LogisticRegression(
            C=float(c_value),
            solver="saga",
            max_iter=LR_MAX_ITER,
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        )
        model.fit(X_train, y_train)

        y_val_prob = model.predict_proba(X_val)[:, 1]
        tuning_rows.append(
            {
                "C": float(c_value),
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            }
        )

    tuning_df = pd.DataFrame(tuning_rows).sort_values("auc", ascending=False).reset_index(drop=True)
    best_c = float(tuning_df.loc[0, "C"] )

    logreg_tfidf_lemma_tuning_results[subset_name] = tuning_df
    logreg_tfidf_lemma_best_c_by_subset[subset_name] = best_c

    print(f"Selected C for subset {subset_name}: {best_c}")
    display(
        tuning_df[["C"] + metric_cols]
        .rename(columns={"average_precision": "pr-auc"})
        .round(6)
    )

    selected_row = tuning_df.loc[0, ["C"] + metric_cols].to_dict()
    tfidf_lemma_validation_rows.append({"subset": subset_name, **selected_row})

    # Refit on the full-train split for final evaluation.
    full_artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    X_train_full = full_artifacts["X_tfidf_lemma_train_full"]
    y_train_full = full_artifacts["y_train_full"]

    final_model = LogisticRegression(
        C=best_c,
        solver="saga",
        max_iter=LR_MAX_ITER,
        random_state=RANDOM_STATE,
        class_weight=CLASS_WEIGHT,
    )
    final_model.fit(X_train_full, y_train_full)
    logreg_tfidf_lemma_models[subset_name] = final_model

tfidf_lemma_validation_metrics_df = pd.DataFrame(tfidf_lemma_validation_rows)
print("Validation metrics for selected C by subset:")
display(
    tfidf_lemma_validation_metrics_df[["subset", "C"] + metric_cols]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

validation_details_tfidf_lemma = tfidf_lemma_validation_metrics_df.copy()
validation_summary_tfidf_lemma = tfidf_lemma_validation_metrics_df[metric_cols].mean().to_frame().T

print("Average validation metrics across subsets:")
display(
    validation_summary_tfidf_lemma.rename(columns={"average_precision": "pr-auc"}).round(6)
)

logreg_tfidf_lemma_best_c = logreg_tfidf_lemma_best_c_by_subset["1"]
logreg_tfidf_lemma_model = logreg_tfidf_lemma_models["1"]


===== Logistic Regression TF-IDF + Lemma | subset 1 (artifacts_1) =====
Using fixed C for subset 1: 0.3


subset 1 C search:   0%|          | 0/1 [00:00<?, ?it/s]

Selected C for subset 1: 0.3


,C,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.3,0.773946,0.56855,0.551146,0.614793,0.470043,0.666073,0.719902



===== Logistic Regression TF-IDF + Lemma | subset 2 (artifacts_2) =====
Using fixed C for subset 2: 0.3


...

,C,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.3,0.773613,0.568027,0.549902,0.612465,0.469901,0.662732,0.719902


Validation metrics for selected C by subset:


,subset,C,auc,pr-auc,f1,f2,precision,recall,accuracy
0,1,0.3,0.773946,0.568550,0.551146,0.614793,0.470043,0.666073,0.719902
1,2,0.3,0.773613,0.568027,0.549902,0.612465,0.469901,0.662732,0.719902


Average validation metrics across subsets:


,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.77378,0.568288,0.550524,0.613629,0.469972,0.664403,0.719902


#### 4.4.3 SBERT

##### Review-level pooling for transformer validation and test splits

In [5]:
from collections import defaultdict
import numpy as np

''' Convolution '''
def smooth_probs_reflect(probs, kernel):
    '''
    Applies convolution kernel over probabilities of adjacent windows
    '''
    
    pad = len(kernel) // 2
    padded = np.pad(probs, pad, mode='reflect')
    return np.convolve(padded, kernel, mode='valid')


'''
Top k = 9 windows are considered
'''


def predict_per_review_topk_smooth(
    y_probs, review_ids, review_label_map, threshold=0.5, k=9
):
    from collections import defaultdict
    import numpy as np

    review_probs = defaultdict(list)

    # Collect probs per review
    for rid, prob in zip(review_ids, y_probs):
        spoiler_prob = prob
        review_probs[rid].append(spoiler_prob)

    review_agg = {}
    KERNEL = np.array([0.2, 0.6, 0.2])

    for rid, probs in review_probs.items():
        probs = np.array(probs)

        smoothed_probs = smooth_probs_reflect(probs, KERNEL)

        topk = np.sort(smoothed_probs)[::-1][:k]
        review_agg[rid] = np.mean(topk)

    sorted_review_ids = sorted(review_agg.keys())

    y_review_scores = [
        review_agg[rid]
        for rid in sorted_review_ids
    ]

    y_review_pred = [
        1 if score > threshold else 0
        for score in y_review_scores
    ]

    y_review_true = [
        review_label_map[rid]
        for rid in sorted_review_ids
    ]

    return y_review_scores, y_review_pred, y_review_true

##### Prediction pipeline

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, fbeta_score,
    precision_score, recall_score, accuracy_score
)

# Shared only across transformer-based Logistic Regression variants.
def run_transformer_logreg_hypertuning(
    X_train_sub,
    X_val,
    y_train_sub,
    val_review_ids,
    val_review_label_map,
    candidate_cs,
    label,
 ):
    tuning_rows = []

    for c_value in tqdm(candidate_cs, desc=label, leave=False):
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=float(c_value),
                solver="saga",
                max_iter=LR_MAX_ITER,
                random_state=RANDOM_STATE,
                class_weight=CLASS_WEIGHT,
            ),
        )
        model.fit(X_train_sub, y_train_sub)

        y_scores, y_pred, y_true = predict_per_review_topk_smooth(
            model.predict_proba(X_val)[:, 1],
            val_review_ids,
            val_review_label_map,
        )

        tuning_rows.append(
            {
                "C": float(c_value),
                "auc": roc_auc_score(y_true, y_scores),
                "average_precision": average_precision_score(y_true, y_scores),
                "f1": f1_score(y_true, y_pred, zero_division=0),
                "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "accuracy": accuracy_score(y_true, y_pred),
            }
        )

    tuning_df = pd.DataFrame(tuning_rows).sort_values("auc", ascending=False).reset_index(drop=True)
    best_c = float(tuning_df.loc[0, "C"] )

    print(f"Selected C: {best_c}")
    display(
        tuning_df[["C", "auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]]
        .rename(columns={"average_precision": "pr-auc"})
        .round(6)
    )

    return best_c, tuning_df

##### Model Training

In [21]:
SUBSET_TO_ARTIFACT = {
    "1": "artifacts_1",
    "2": "artifacts_2",
}

metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
logreg_sbert_best_c_by_subset = {}
logreg_sbert_models = {}
logreg_sbert_tuning_results = {}
sbert_validation_rows = []

for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    print(f"\n===== Logistic Regression SBERT | subset {subset_name} ({artifact_name}) =====")

    # SBERT validation split for model selection.
    validation_data = load_transformer_validation_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    X_sbert_train_sub = validation_data["X_train"]
    y_sbert_train_sub = validation_data["y_train"]
    X_sbert_val = validation_data["X_val"]
    review_ids_val = validation_data["review_ids_val"]
    review_label_map_val = validation_data["review_label_map_val"]

    if X_sbert_train_sub.shape[0] != len(y_sbert_train_sub):
        raise ValueError(
            f"SBERT artifact mismatch for {artifact_name}: train={X_sbert_train_sub.shape[0]}/{len(y_sbert_train_sub)}"
        )

    # SBERT hyperparameter search on the validation split.
    artifact_idx = int(subset_name) - 1
    fixed_c = LR_SBERT_FIXED_C[artifact_idx] if artifact_idx < len(LR_SBERT_FIXED_C) else None

    if USE_FIXED_BEST_PARAMS and fixed_c is not None:
        candidate_cs = [float(fixed_c)]
        print(f"Using fixed C for subset {subset_name}: {candidate_cs[0]}")
    else:
        candidate_cs = SBERT_LR_C_GRID

    best_c_sbert, tuning_df = run_transformer_logreg_hypertuning(
        X_sbert_train_sub,
        X_sbert_val,
        y_sbert_train_sub,
        review_ids_val,
        review_label_map_val,
        candidate_cs,
        label=f"SBERT subset {subset_name} C search",
    )

    logreg_sbert_best_c_by_subset[subset_name] = best_c_sbert
    logreg_sbert_tuning_results[subset_name] = tuning_df

    selected_row = tuning_df.loc[0, ["C"] + metric_cols].to_dict()
    sbert_validation_rows.append({"subset": subset_name, **selected_row})

    # SBERT full-train refit for final evaluation.
    fulltrain_data = load_transformer_fulltrain_test_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    X_sbert_train_full = fulltrain_data["X_train_full"]
    y_sbert_train_full = fulltrain_data["y_train_full"]

    final_model_sbert = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=best_c_sbert,
            solver="saga",
            max_iter=LR_MAX_ITER,
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        ),
    )
    final_model_sbert.fit(X_sbert_train_full, y_sbert_train_full)
    logreg_sbert_models[subset_name] = final_model_sbert

sbert_validation_metrics_df = pd.DataFrame(sbert_validation_rows)
print("Validation metrics for selected C by subset:")
display(
    sbert_validation_metrics_df[["subset", "C"] + metric_cols]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

validation_details_sbert = sbert_validation_metrics_df.copy()
validation_summary_sbert = sbert_validation_metrics_df[metric_cols].mean().to_frame().T
print("Average validation metrics across subsets:")
display(
    validation_summary_sbert.rename(columns={"average_precision": "pr-auc"}).round(6)
)

logreg_sbert_best_c = logreg_sbert_best_c_by_subset["1"]
logreg_sbert_model = logreg_sbert_models["1"]


===== Logistic Regression SBERT | subset 1 (artifacts_1) =====
Using fixed C for subset 1: 0.3


SBERT subset 1 C search:   0%|          | 0/1 [00:00<?, ?it/s]

Selected C: 0.3


,C,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.3,0.736767,0.495091,0.500012,0.621383,0.377214,0.741352,0.6298



===== Logistic Regression SBERT | subset 2 (artifacts_2) =====
Using fixed C for subset 2: 10.0


...

,C,auc,pr-auc,f1,f2,precision,recall,accuracy
0,10.0,0.736921,0.494959,0.500157,0.621836,0.377156,0.742213,0.629585


Validation metrics for selected C by subset:


,subset,C,auc,pr-auc,f1,f2,precision,recall,accuracy
0,1,0.3,0.736767,0.495091,0.500012,0.621383,0.377214,0.741352,0.629800
1,2,10.0,0.736921,0.494959,0.500157,0.621836,0.377156,0.742213,0.629585


Average validation metrics across subsets:


,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.736844,0.495025,0.500085,0.62161,0.377185,0.741783,0.629693


#### 4.4.4 RoBERTa

In [ ]:
SUBSET_TO_ARTIFACT = {
    "1": "artifacts_1",
    "2": "artifacts_2",
}

metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
logreg_roberta_best_c_by_subset = {}
logreg_roberta_models = {}
logreg_roberta_tuning_results = {}
roberta_validation_rows = []

for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    print(f"\n===== Logistic Regression RoBERTa | subset {subset_name} ({artifact_name}) =====")

    # RoBERTa validation split for model selection.
    validation_data = load_transformer_validation_splits(
        artifact_name=artifact_name,
        model_type="roberta",
    )
    X_roberta_train_sub = validation_data["X_train"]
    y_roberta_train_sub = validation_data["y_train"]
    X_roberta_val = validation_data["X_val"]
    review_ids_val = validation_data["review_ids_val"]
    review_label_map_val = validation_data["review_label_map_val"]

    if X_roberta_train_sub.shape[0] != len(y_roberta_train_sub):
        raise ValueError(
            f"RoBERTa artifact mismatch for {artifact_name}: train={X_roberta_train_sub.shape[0]}/{len(y_roberta_train_sub)}"
        )

    # RoBERTa hyperparameter search on the validation split.
    artifact_idx = int(subset_name) - 1
    fixed_c = LR_ROBERTA_FIXED_C[artifact_idx] if artifact_idx < len(LR_ROBERTA_FIXED_C) else None

    if USE_FIXED_BEST_PARAMS and fixed_c is not None:
        candidate_cs = [float(fixed_c)]
        print(f"Using fixed C for subset {subset_name}: {candidate_cs[0]}")
    else:
        candidate_cs = ROBERTA_LR_C_GRID

    best_c_roberta, tuning_df = run_transformer_logreg_hypertuning(
        X_roberta_train_sub,
        X_roberta_val,
        y_roberta_train_sub,
        review_ids_val,
        review_label_map_val,
        candidate_cs,
        label=f"RoBERTa subset {subset_name} C search",
    )

    logreg_roberta_best_c_by_subset[subset_name] = best_c_roberta
    logreg_roberta_tuning_results[subset_name] = tuning_df

    selected_row = tuning_df.loc[0, ["C"] + metric_cols].to_dict()
    roberta_validation_rows.append({"subset": subset_name, **selected_row})

    # RoBERTa full-train refit for final evaluation.
    fulltrain_data = load_transformer_fulltrain_test_splits(
        artifact_name=artifact_name,
        model_type="roberta",
    )
    X_roberta_train_full = fulltrain_data["X_train_full"]
    y_roberta_train_full = fulltrain_data["y_train_full"]

    final_model_roberta = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=best_c_roberta,
            solver="saga",
            max_iter=LR_MAX_ITER,
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        ),
    )
    final_model_roberta.fit(X_roberta_train_full, y_roberta_train_full)
    logreg_roberta_models[subset_name] = final_model_roberta

roberta_validation_metrics_df = pd.DataFrame(roberta_validation_rows)
print("Validation metrics for selected C by subset:")
display(
    roberta_validation_metrics_df[["subset", "C"] + metric_cols]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

validation_details_roberta = roberta_validation_metrics_df.copy()
validation_summary_roberta = roberta_validation_metrics_df[metric_cols].mean().to_frame().T
print("Average validation metrics across subsets:")
display(
    validation_summary_roberta.rename(columns={"average_precision": "pr-auc"}).round(6)
)

logreg_roberta_best_c = logreg_roberta_best_c_by_subset["1"]
logreg_roberta_model = logreg_roberta_models["1"]

#### 4.4.5 Validation Summary

In [ ]:
LOGREG_METRIC_COLUMNS = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]

validation_summaries = {
    "TF-IDF": validation_summary_tfidf,
    "TF-IDF+Lemma": validation_summary_tfidf_lemma,
    "SBERT": validation_summary_sbert,
    "RoBERTa": validation_summary_roberta,
}

logreg_train_metrics_df = pd.DataFrame(
    [
        {"feature": feature_name, **summary_df.iloc[0].to_dict()}
        for feature_name, summary_df in validation_summaries.items()
    ]
).rename(columns={"average_precision": "pr-auc"})

print("Logistic Regression validation metrics:")
display(
    logreg_train_metrics_df[["feature", "auc", "pr-auc", "f1", "f2", "precision", "recall", "accuracy"]]
    .round(6)
)

NameError: name 'validation_summary_tfidf' is not defined

### 4.5 Support Vector Machine

#### 4.5.1 TF-IDF

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

SUBSET_TO_ARTIFACT = {
    "1": "artifacts_1",
    "2": "artifacts_2",
}

metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
svm_tfidf_best_c_by_subset = {}
svm_tfidf_models = {}
svm_tfidf_tuning_results = {}
tfidf_validation_rows = []

for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    print(f"\n===== SVM TF-IDF | subset {subset_name} ({artifact_name}) =====")

    # Validation split for model selection.
    validation_artifacts = load_validation_split(artifact_name=artifact_name)
    X_train = validation_artifacts["X_tfidf_train"]
    X_val = validation_artifacts["X_tfidf_val"]
    y_train = validation_artifacts["y_train"]
    y_val = validation_artifacts["y_val"]

    if X_train.shape[0] != len(y_train) or X_val.shape[0] != len(y_val):
        raise ValueError(
            f"TF-IDF artifact mismatch for {artifact_name}: "
            f"train={X_train.shape[0]}/{len(y_train)}, val={X_val.shape[0]}/{len(y_val)}"
        )

    artifact_idx = int(subset_name) - 1
    fixed_c = SVM_TFIDF_FIXED_C[artifact_idx] if artifact_idx < len(SVM_TFIDF_FIXED_C) else None

    if USE_FIXED_BEST_PARAMS and fixed_c is not None:
        candidate_cs = [float(fixed_c)]
        print(f"Using fixed C for subset {subset_name}: {candidate_cs[0]}")
    else:
        candidate_cs = SVM_C_GRID

    tuning_rows = []
    for c_value in tqdm(candidate_cs, desc=f"subset {subset_name} C search", leave=False):
        base_model = LinearSVC(
            C=float(c_value),
            tol=SVM_TOL,
            max_iter=SVM_MAX_ITER,
            dual="auto",
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        )
        model = CalibratedClassifierCV(
            base_model,
            cv=SVM_CALIBRATION_CV,
            method=SVM_CALIBRATION_METHOD,
        )
        model.fit(X_train, y_train)

        y_val_prob = model.predict_proba(X_val)[:, 1]
        tuning_rows.append(
            {
                "C": float(c_value),
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            }
        )

    tuning_df = pd.DataFrame(tuning_rows).sort_values("auc", ascending=False).reset_index(drop=True)
    best_c = float(tuning_df.loc[0, "C"])

    svm_tfidf_tuning_results[subset_name] = tuning_df
    svm_tfidf_best_c_by_subset[subset_name] = best_c

    print(f"Selected C for subset {subset_name}: {best_c}")
    display(
        tuning_df[["C"] + metric_cols]
        .rename(columns={"average_precision": "pr-auc"})
        .round(6)
    )

    selected_row = tuning_df.loc[0, ["C"] + metric_cols].to_dict()
    tfidf_validation_rows.append({"subset": subset_name, **selected_row})

    # Refit on the full-train split for final evaluation.
    full_artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    X_train_full = full_artifacts["X_tfidf_train_full"]
    y_train_full = full_artifacts["y_train_full"]

    final_base_model = LinearSVC(
        C=best_c,
        tol=SVM_TOL,
        max_iter=SVM_MAX_ITER,
        dual="auto",
        random_state=RANDOM_STATE,
        class_weight=CLASS_WEIGHT,
    )
    final_model = CalibratedClassifierCV(
        final_base_model,
        cv=SVM_CALIBRATION_CV,
        method=SVM_CALIBRATION_METHOD,
    )
    final_model.fit(X_train_full, y_train_full)
    svm_tfidf_models[subset_name] = final_model

tfidf_validation_metrics_df = pd.DataFrame(tfidf_validation_rows)
print("Validation metrics for selected C by subset:")
display(
    tfidf_validation_metrics_df[["subset", "C"] + metric_cols]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

validation_details_svm_tfidf = tfidf_validation_metrics_df.copy()
validation_summary_svm_tfidf = tfidf_validation_metrics_df[metric_cols].mean().to_frame().T
print("Average validation metrics across subsets:")
display(
    validation_summary_svm_tfidf.rename(columns={"average_precision": "pr-auc"}).round(6)
)

svm_tfidf_best_c = svm_tfidf_best_c_by_subset["1"]
svm_tfidf_model = svm_tfidf_models["1"]

# Keep the section-level aliases available while the rest of the notebook is being aligned.
validation_details_svm = validation_details_svm_tfidf
validation_summary_svm = validation_summary_svm_tfidf


===== SVM TF-IDF | subset 1 (artifacts_1) =====


subset 1 C search:   0%|          | 0/7 [00:00<?, ?it/s]


===== SVM TF-IDF | subset 1 (artifacts_1) =====


subset 1 C search:   0%|          | 0/7 [00:00<?, ?it/s]

Selected C for subset 1: 0.1


...

,C,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.10,0.773772,0.568078,0.535706,0.530990,0.543755,0.527892,0.763756
1,0.03,0.771919,0.560666,0.533076,0.532069,0.534764,0.531400,0.759659
2,0.30,0.766304,0.562002,0.525935,0.516270,0.542874,0.510021,0.762620
3,0.01,0.763094,0.543130,0.521400,0.520886,0.522259,0.520543,0.753277
4,1.00,0.750868,0.544540,0.505696,0.489619,0.534973,0.479457,0.758006
5,3.00,0.736499,0.526412,0.484423,0.461608,0.527909,0.447556,0.754039
6,10.00,0.725260,0.511568,0.463350,0.434629,0.520697,0.417381,0.750388


Validation metrics for selected C by subset:


,subset,C,auc,pr-auc,f1,f2,precision,recall,accuracy
0,1,0.1,0.774289,0.568790,0.535665,0.531447,0.542846,0.528672,0.763367
1,2,0.1,0.773772,0.568078,0.535706,0.530990,0.543755,0.527892,0.763756


Average validation metrics across subsets:


,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.774031,0.568434,0.535686,0.531219,0.5433,0.528282,0.763561


#### 4.5.2 SBERT

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SUBSET_TO_ARTIFACT = {
    "1": "artifacts_1",
    "2": "artifacts_2",
}

metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
svm_sbert_best_c_by_subset = {}
svm_sbert_models = {}
svm_sbert_tuning_results = {}
sbert_validation_rows = []

for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    print(f"\n===== SVM SBERT | subset {subset_name} ({artifact_name}) =====")

    # SBERT validation split for model selection.
    validation_data = load_transformer_validation_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    X_sbert_train_sub = validation_data["X_train"]
    y_sbert_train_sub = validation_data["y_train"]
    X_sbert_val = validation_data["X_val"]
    review_ids_val = validation_data["review_ids_val"]
    review_label_map_val = validation_data["review_label_map_val"]

    if X_sbert_train_sub.shape[0] != len(y_sbert_train_sub):
        raise ValueError(
            f"SBERT artifact mismatch for {artifact_name}: train={X_sbert_train_sub.shape[0]}/{len(y_sbert_train_sub)}"
        )

    artifact_idx = int(subset_name) - 1
    fixed_c = SVM_SBERT_FIXED_C[artifact_idx] if artifact_idx < len(SVM_SBERT_FIXED_C) else None

    if USE_FIXED_BEST_PARAMS and fixed_c is not None:
        candidate_cs = [float(fixed_c)]
        print(f"Using fixed C for subset {subset_name}: {candidate_cs[0]}")
    else:
        candidate_cs = SVM_C_GRID

    tuning_rows = []
    for c_value in tqdm(candidate_cs, desc=f"SBERT subset {subset_name} C search", leave=False):
        base_model = make_pipeline(
            StandardScaler(),
            LinearSVC(
                C=float(c_value),
                tol=SVM_TOL,
                max_iter=SVM_MAX_ITER,
                dual="auto",
                random_state=RANDOM_STATE,
                class_weight=CLASS_WEIGHT,
            ),
        )
        model = CalibratedClassifierCV(
            base_model,
            cv=SVM_CALIBRATION_CV,
            method=SVM_CALIBRATION_METHOD,
        )
        model.fit(X_sbert_train_sub, y_sbert_train_sub)

        y_scores, y_pred, y_true = predict_per_review_topk_smooth(
            model.predict_proba(X_sbert_val)[:, 1],
            review_ids_val,
            review_label_map_val,
        )
        tuning_rows.append(
            {
                "C": float(c_value),
                "auc": roc_auc_score(y_true, y_scores),
                "average_precision": average_precision_score(y_true, y_scores),
                "f1": f1_score(y_true, y_pred, zero_division=0),
                "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "accuracy": accuracy_score(y_true, y_pred),
            }
        )

    tuning_df = pd.DataFrame(tuning_rows).sort_values("auc", ascending=False).reset_index(drop=True)
    best_c = float(tuning_df.loc[0, "C"])

    svm_sbert_tuning_results[subset_name] = tuning_df
    svm_sbert_best_c_by_subset[subset_name] = best_c

    print(f"Selected C for subset {subset_name}: {best_c}")
    display(
        tuning_df[["C"] + metric_cols]
        .rename(columns={"average_precision": "pr-auc"})
        .round(6)
    )

    selected_row = tuning_df.loc[0, ["C"] + metric_cols].to_dict()
    sbert_validation_rows.append({"subset": subset_name, **selected_row})

    # Refit on the full-train split for final evaluation.
    fulltrain_data = load_transformer_fulltrain_test_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    X_sbert_train_full = fulltrain_data["X_train_full"]
    y_sbert_train_full = fulltrain_data["y_train_full"]

    final_base_model = make_pipeline(
        StandardScaler(),
        LinearSVC(
            C=best_c,
            tol=SVM_TOL,
            max_iter=SVM_MAX_ITER,
            dual="auto",
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        ),
    )
    final_model = CalibratedClassifierCV(
        final_base_model,
        cv=SVM_CALIBRATION_CV,
        method=SVM_CALIBRATION_METHOD,
    )
    final_model.fit(X_sbert_train_full, y_sbert_train_full)
    svm_sbert_models[subset_name] = final_model

sbert_validation_metrics_df = pd.DataFrame(sbert_validation_rows)
print("Validation metrics for selected C by subset:")
display(
    sbert_validation_metrics_df[["subset", "C"] + metric_cols]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

validation_details_svm_sbert = sbert_validation_metrics_df.copy()
validation_summary_svm_sbert = sbert_validation_metrics_df[metric_cols].mean().to_frame().T
print("Average validation metrics across subsets:")
display(
    validation_summary_svm_sbert.rename(columns={"average_precision": "pr-auc"}).round(6)
)

svm_sbert_best_c = svm_sbert_best_c_by_subset["1"]
svm_sbert_model = svm_sbert_models["1"]

#### 4.5.3 Validation Summary

In [ ]:
SVM_METRIC_COLUMNS = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]

validation_summaries = {
    "TF-IDF": validation_summary_svm_tfidf,
    "SBERT": validation_summary_svm_sbert,
}

svm_train_metrics_df = pd.DataFrame(
    [
        {"feature": feature_name, **summary_df.iloc[0].to_dict()}
        for feature_name, summary_df in validation_summaries.items()
    ]
).rename(columns={"average_precision": "pr-auc"})

print("SVM validation metrics:")
display(
    svm_train_metrics_df[["feature", "auc", "pr-auc", "f1", "f2", "precision", "recall", "accuracy"]]
    .round(6)
)

### 4.6 Decision Tree

#### 4.6.1 TF-IDF

In [7]:
DT_INT_PARAMS = ["max_depth", "min_samples_split", "min_samples_leaf"]
DT_FLOAT_PARAMS = ["ccp_alpha"]

def normalize_dt_params(params_like):
    params = dict(params_like)
    for name in DT_INT_PARAMS:
        value = params.get(name)
        params[name] = None if value is None or pd.isna(value) else int(value)
    for name in DT_FLOAT_PARAMS:
        value = params.get(name)
        params[name] = float(value)
    return params

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import ParameterSampler

SUBSET_TO_ARTIFACT = {
    "1": "artifacts_1",
    "2": "artifacts_2",
}

metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
dt_param_cols = ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"]
dt_tfidf_best_params_by_subset = {}
dt_tfidf_models = {}
dt_tfidf_tuning_results = {}
tfidf_validation_rows = []

for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    print(f"\n===== Decision Tree TF-IDF | subset {subset_name} ({artifact_name}) =====")

    validation_artifacts = load_validation_split(artifact_name=artifact_name)
    X_train = validation_artifacts["X_tfidf_train"]
    X_val = validation_artifacts["X_tfidf_val"]
    y_train = validation_artifacts["y_train"]
    y_val = validation_artifacts["y_val"]

    if X_train.shape[0] != len(y_train) or X_val.shape[0] != len(y_val):
        raise ValueError(
            f"TF-IDF artifact mismatch for {artifact_name}: "
            f"train={X_train.shape[0]}/{len(y_train)}, val={X_val.shape[0]}/{len(y_val)}"
        )

    if USE_FIXED_BEST_PARAMS and DT_TFIDF_FIXED_PARAMS is not None:
        candidate_param_sets = [normalize_dt_params(DT_TFIDF_FIXED_PARAMS)]
        print(f"Using fixed params for subset {subset_name}: {candidate_param_sets[0]}")
    else:
        candidate_param_sets = [
            normalize_dt_params(params)
            for params in ParameterSampler(
                DT_PARAM_GRID,
                n_iter=min(DT_RANDOM_SEARCH_ITER, int(np.prod([len(values) for values in DT_PARAM_GRID.values()]))),
                random_state=RANDOM_STATE,
            )
        ]

    tuning_rows = []
    for candidate_params in tqdm(candidate_param_sets, desc=f"DT TF-IDF search {subset_name}", leave=False):
        model = DecisionTreeClassifier(
            **candidate_params,
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        )
        model.fit(X_train, y_train)

        y_val_prob = model.predict_proba(X_val)[:, 1]
        tuning_rows.append(
            {
                **candidate_params,
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            }
        )

    tuning_df = pd.DataFrame(tuning_rows).sort_values("auc", ascending=False).reset_index(drop=True)
    best_params = normalize_dt_params(tuning_df.loc[0, dt_param_cols])

    dt_tfidf_tuning_results[subset_name] = tuning_df
    dt_tfidf_best_params_by_subset[subset_name] = best_params

    print(f"Selected params for subset {subset_name}: {best_params}")
    display(
        tuning_df[dt_param_cols + metric_cols]
        .rename(columns={"average_precision": "pr-auc"})
        .round(6)
    )

    selected_row = tuning_df.loc[0, dt_param_cols + metric_cols].to_dict()
    tfidf_validation_rows.append({"subset": subset_name, **selected_row})

    full_artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    X_train_full = full_artifacts["X_tfidf_train_full"]
    y_train_full = full_artifacts["y_train_full"]

    final_model = DecisionTreeClassifier(
        **best_params,
        random_state=RANDOM_STATE,
        class_weight=CLASS_WEIGHT,
    )
    final_model.fit(X_train_full, y_train_full)
    dt_tfidf_models[subset_name] = final_model

tfidf_validation_metrics_df = pd.DataFrame(tfidf_validation_rows)
print("Validation metrics for selected params by subset:")
display(
    tfidf_validation_metrics_df[["subset"] + dt_param_cols + metric_cols]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

validation_details_dt_tfidf = tfidf_validation_metrics_df.copy()
validation_summary_dt_tfidf = tfidf_validation_metrics_df[metric_cols].mean().to_frame().T
print("Average validation metrics across subsets:")
display(
    validation_summary_dt_tfidf.rename(columns={"average_precision": "pr-auc"}).round(6)
)

dt_tfidf_best_params = dt_tfidf_best_params_by_subset["1"]
dt_tfidf_model = dt_tfidf_models["1"]

#### 4.6.2 SBERT

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import ParameterSampler

SUBSET_TO_ARTIFACT = {
    "1": "artifacts_1",
    "2": "artifacts_2",
}

metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
dt_param_cols = ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"]
dt_sbert_best_params_by_subset = {}
dt_sbert_models = {}
dt_sbert_tuning_results = {}
sbert_validation_rows = []

for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    print(f"\n===== Decision Tree SBERT | subset {subset_name} ({artifact_name}) =====")

    validation_data = load_transformer_validation_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    X_sbert_train_sub = validation_data["X_train"]
    y_sbert_train_sub = validation_data["y_train"]
    X_sbert_val = validation_data["X_val"]
    review_ids_val = validation_data["review_ids_val"]
    review_label_map_val = validation_data["review_label_map_val"]

    if X_sbert_train_sub.shape[0] != len(y_sbert_train_sub):
        raise ValueError(
            f"SBERT artifact mismatch for {artifact_name}: train={X_sbert_train_sub.shape[0]}/{len(y_sbert_train_sub)}"
        )

    if USE_FIXED_BEST_PARAMS and DT_SBERT_FIXED_PARAMS is not None:
        candidate_param_sets = [normalize_dt_params(DT_SBERT_FIXED_PARAMS)]
        print(f"Using fixed params for subset {subset_name}: {candidate_param_sets[0]}")
    else:
        candidate_param_sets = [
            normalize_dt_params(params)
            for params in ParameterSampler(
                DT_PARAM_GRID,
                n_iter=min(DT_RANDOM_SEARCH_ITER, int(np.prod([len(values) for values in DT_PARAM_GRID.values()]))),
                random_state=RANDOM_STATE,
            )
        ]

    tuning_rows = []
    for candidate_params in tqdm(candidate_param_sets, desc=f"DT SBERT search {subset_name}", leave=False):
        model = DecisionTreeClassifier(
            **candidate_params,
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        )
        model.fit(X_sbert_train_sub, y_sbert_train_sub)

        y_scores, y_pred, y_true = predict_per_review_topk_smooth(
            model.predict_proba(X_sbert_val)[:, 1],
            review_ids_val,
            review_label_map_val,
        )
        tuning_rows.append(
            {
                **candidate_params,
                "auc": roc_auc_score(y_true, y_scores),
                "average_precision": average_precision_score(y_true, y_scores),
                "f1": f1_score(y_true, y_pred, zero_division=0),
                "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "accuracy": accuracy_score(y_true, y_pred),
            }
        )

    tuning_df = pd.DataFrame(tuning_rows).sort_values("auc", ascending=False).reset_index(drop=True)
    best_params = normalize_dt_params(tuning_df.loc[0, dt_param_cols])

    dt_sbert_tuning_results[subset_name] = tuning_df
    dt_sbert_best_params_by_subset[subset_name] = best_params

    print(f"Selected params for subset {subset_name}: {best_params}")
    display(
        tuning_df[dt_param_cols + metric_cols]
        .rename(columns={"average_precision": "pr-auc"})
        .round(6)
    )

    selected_row = tuning_df.loc[0, dt_param_cols + metric_cols].to_dict()
    sbert_validation_rows.append({"subset": subset_name, **selected_row})

    fulltrain_data = load_transformer_fulltrain_test_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    X_sbert_train_full = fulltrain_data["X_train_full"]
    y_sbert_train_full = fulltrain_data["y_train_full"]

    final_model = DecisionTreeClassifier(
        **best_params,
        random_state=RANDOM_STATE,
        class_weight=CLASS_WEIGHT,
    )
    final_model.fit(X_sbert_train_full, y_sbert_train_full)
    dt_sbert_models[subset_name] = final_model

sbert_validation_metrics_df = pd.DataFrame(sbert_validation_rows)
print("Validation metrics for selected params by subset:")
display(
    sbert_validation_metrics_df[["subset"] + dt_param_cols + metric_cols]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

validation_details_dt_sbert = sbert_validation_metrics_df.copy()
validation_summary_dt_sbert = sbert_validation_metrics_df[metric_cols].mean().to_frame().T
print("Average validation metrics across subsets:")
display(
    validation_summary_dt_sbert.rename(columns={"average_precision": "pr-auc"}).round(6)
)

dt_sbert_best_params = dt_sbert_best_params_by_subset["1"]
dt_sbert_model = dt_sbert_models["1"]

#### 4.6.3 Validation Summary

In [ ]:
DT_METRIC_COLUMNS = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]

validation_summaries = {
    "TF-IDF": validation_summary_dt_tfidf,
    "SBERT": validation_summary_dt_sbert,
}

dt_train_metrics_df = pd.DataFrame(
    [
        {"feature": feature_name, **summary_df.iloc[0].to_dict()}
        for feature_name, summary_df in validation_summaries.items()
    ]
).rename(columns={"average_precision": "pr-auc"})

print("Decision Tree validation metrics:")
display(
    dt_train_metrics_df[["feature", "auc", "pr-auc", "f1", "f2", "precision", "recall", "accuracy"]]
    .round(6)
)

### 4.7 Random Forest

#### 4.7.1 TF-IDF

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    fbeta_score, precision_score, recall_score, accuracy_score
)

RF_THRESHOLD = 0.5
SUBSET_TO_ARTIFACT = {"1": "artifacts_1", "2": "artifacts_2"}
metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
rf_param_cols = ["n_estimators", "max_depth", "min_samples_split", "min_samples_leaf", "max_features"]


def normalize_rf_params(params_like):
    params = {key: value for key, value in dict(params_like).items() if key in rf_param_cols}
    for name in ["n_estimators", "max_depth", "min_samples_split", "min_samples_leaf"]:
        if name in params and params[name] is not None:
            params[name] = int(params[name])
    return params


rf_tfidf_best_params_by_subset = {}
rf_tfidf_models = {}
rf_tfidf_tuning_results = {}
rf_tfidf_validation_rows = []

for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    print(f"\n===== Random Forest TF-IDF | subset {subset_name} ({artifact_name}) =====")

    validation_artifacts = load_validation_split(artifact_name=artifact_name)
    X_train = validation_artifacts["X_tfidf_train"]
    X_val = validation_artifacts["X_tfidf_val"]
    y_train = np.array(validation_artifacts["y_train"]).ravel()
    y_val = np.array(validation_artifacts["y_val"]).ravel()

    if X_train.shape[0] != len(y_train) or X_val.shape[0] != len(y_val):
        raise ValueError(
            f"TF-IDF artifact mismatch for {artifact_name}: "
            f"train={X_train.shape[0]}/{len(y_train)}, val={X_val.shape[0]}/{len(y_val)}"
        )

    if USE_FIXED_BEST_PARAMS and globals().get("RF_TFIDF_FIXED_PARAMS") is not None:
        candidate_param_sets = [normalize_rf_params(RF_TFIDF_FIXED_PARAMS)]
        print(f"Using fixed params for subset {subset_name}: {candidate_param_sets[0]}")
    else:
        candidate_param_sets = [
            normalize_rf_params(params)
            for params in ParameterSampler(
                RF_PARAM_GRID,
                n_iter=RF_RANDOM_SEARCH_ITER,
                random_state=RANDOM_STATE,
            )
        ]

    tuning_rows = []
    for candidate_params in tqdm(candidate_param_sets, desc=f"RF TF-IDF search {subset_name}", leave=False):
        model = RandomForestClassifier(
            **candidate_params,
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
            n_jobs=-1,
        )
        model.fit(X_train, y_train)

        y_val_prob = model.predict_proba(X_val)[:, 1]
        tuning_rows.append(
            {
                **candidate_params,
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= RF_THRESHOLD, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= RF_THRESHOLD, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= RF_THRESHOLD, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= RF_THRESHOLD, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= RF_THRESHOLD),
            }
        )
        gc.collect()

    tuning_df = pd.DataFrame(tuning_rows).sort_values("auc", ascending=False).reset_index(drop=True)
    best_params = normalize_rf_params(tuning_df.iloc[0])
    present_param_cols = [column for column in rf_param_cols if column in tuning_df.columns]

    rf_tfidf_tuning_results[subset_name] = tuning_df
    rf_tfidf_best_params_by_subset[subset_name] = best_params

    print(f"Selected RF parameters for subset {subset_name}: {best_params}")
    display(
        tuning_df[present_param_cols + metric_cols]
        .rename(columns={"average_precision": "pr-auc"})
        .round(6)
    )

    selected_row = tuning_df.loc[0, present_param_cols + metric_cols].to_dict()
    rf_tfidf_validation_rows.append({"subset": subset_name, **selected_row})

    full_artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    X_train_full = full_artifacts["X_tfidf_train_full"]
    y_train_full = np.array(full_artifacts["y_train_full"]).ravel()

    final_model = RandomForestClassifier(
        **best_params,
        random_state=RANDOM_STATE,
        class_weight=CLASS_WEIGHT,
        n_jobs=-1,
    )
    final_model.fit(X_train_full, y_train_full)
    rf_tfidf_models[subset_name] = final_model

    del validation_artifacts, full_artifacts
    gc.collect()

rf_tfidf_validation_metrics_df = pd.DataFrame(rf_tfidf_validation_rows)
rf_tfidf_present_param_cols = [column for column in rf_param_cols if column in rf_tfidf_validation_metrics_df.columns]
print("Validation metrics for selected params by subset:")
display(
    rf_tfidf_validation_metrics_df[["subset"] + rf_tfidf_present_param_cols + metric_cols]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

validation_details_rf_tfidf = rf_tfidf_validation_metrics_df.copy()
validation_summary_rf_tfidf = rf_tfidf_validation_metrics_df[metric_cols].mean().to_frame().T
print("Average validation metrics across subsets:")
display(
    validation_summary_rf_tfidf.rename(columns={"average_precision": "pr-auc"}).round(6)
)

rf_tfidf_best_params = rf_tfidf_best_params_by_subset["1"]
rf_tfidf_model = rf_tfidf_models["1"]

#### 4.7.2 Threshold Analysis and Confusion Matrices

In [ ]:
from sklearn.metrics import confusion_matrix, fbeta_score, accuracy_score

thresholds = np.arange(0.1, 0.95, 0.05)
threshold_rows_by_subset = []

for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    y_test = np.array(artifacts["y_test"]).ravel()
    X_test_tfidf = artifacts["X_tfidf_test"]

    y_test_prob_dt = dt_tfidf_models[subset_name].predict_proba(X_test_tfidf)[:, 1]
    y_test_prob_rf = rf_tfidf_models[subset_name].predict_proba(X_test_tfidf)[:, 1]

    subset_rows = []
    for threshold in thresholds:
        y_pred_rf = y_test_prob_rf >= threshold
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred_rf).ravel()

        subset_rows.append(
            {
                "subset": subset_name,
                "Threshold": round(threshold, 2),
                "FPR (Minimize)": fp / (fp + tn),
                "Recall": tp / (tp + fn),
                "F2-Score": fbeta_score(y_test, y_pred_rf, beta=2, zero_division=0),
                "Accuracy": accuracy_score(y_test, y_pred_rf),
                "DT Accuracy @0.5": accuracy_score(y_test, y_test_prob_dt >= 0.5),
            }
        )

    threshold_rows_by_subset.append(pd.DataFrame(subset_rows))

threshold_df = (
    pd.concat(threshold_rows_by_subset, ignore_index=True)
    .groupby("Threshold", as_index=False)[["FPR (Minimize)", "Recall", "F2-Score", "Accuracy", "DT Accuracy @0.5"]]
    .mean()
)

print("\n--- Threshold Optimization Table (Random Forest, averaged across subsets) ---")
display(threshold_df.sort_values("Threshold").round(6))

In [ ]:
from sklearn.metrics import confusion_matrix, fbeta_score, accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

threshold_dt = 0.5
thresholds_rf = [0.45, 0.5, 0.55]

results = {
    "DT_0.5": {"cm": [], "fpr": [], "f2": []},
    "RF_0.45": {"cm": [], "fpr": [], "f2": []},
    "RF_0.5": {"cm": [], "fpr": [], "f2": []},
    "RF_0.55": {"cm": [], "fpr": [], "f2": []},
}

for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    artifacts = load_validation_split(artifact_name=artifact_name)
    X_train = artifacts["X_tfidf_train"]
    y_train = np.array(artifacts["y_train"]).ravel()
    X_val = artifacts["X_tfidf_val"]
    y_val = np.array(artifacts["y_val"]).ravel()

    dt = DecisionTreeClassifier(
        **dt_tfidf_best_params_by_subset[subset_name],
        random_state=RANDOM_STATE,
        class_weight=CLASS_WEIGHT,
    ).fit(X_train, y_train)
    rf = RandomForestClassifier(
        **rf_tfidf_best_params_by_subset[subset_name],
        random_state=RANDOM_STATE,
        class_weight=CLASS_WEIGHT,
        n_jobs=-1,
    ).fit(X_train, y_train)

    prob_dt = dt.predict_proba(X_val)[:, 1]
    prob_rf = rf.predict_proba(X_val)[:, 1]

    def add_fold_data(key, y_true, y_prob, threshold):
        y_pred = y_prob >= threshold
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, _, _ = cm.ravel()
        results[key]["cm"].append(cm)
        results[key]["fpr"].append(fp / (fp + tn))
        results[key]["f2"].append(fbeta_score(y_true, y_pred, beta=2, zero_division=0))

    add_fold_data("DT_0.5", y_val, prob_dt, threshold_dt)
    for threshold in thresholds_rf:
        add_fold_data(f"RF_{threshold}", y_val, prob_rf, threshold)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

plot_configs = [
    ("Decision Tree", "DT_0.5", 0.5),
    ("Random Forest", "RF_0.45", 0.45),
    ("Random Forest", "RF_0.5", 0.5),
    ("Random Forest", "RF_0.55", 0.55),
]

for i, (title, key, threshold) in enumerate(plot_configs):
    avg_cm = np.mean(results[key]["cm"], axis=0)
    avg_fpr = np.mean(results[key]["fpr"])
    avg_f2 = np.mean(results[key]["f2"])

    sns.heatmap(avg_cm, annot=True, fmt='.1f', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f"{title} (Threshold {threshold})\nAvg FPR: {avg_fpr:.3f} | Avg F2: {avg_f2:.3f}", fontsize=14)
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('Actual Label')
    axes[i].set_xticklabels(['Safe', 'Spoiler'])
    axes[i].set_yticklabels(['Safe', 'Spoiler'])

plt.tight_layout()
plt.show()

#### 4.7.3 SBERT

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, 
    fbeta_score, precision_score, recall_score, accuracy_score
)

# --- CONFIGURATION ---
RF_THRESHOLD = 0.5
SUBSET_TO_ARTIFACT = {"1": "artifacts_1", "2": "artifacts_2"}
metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]

# Only these keys will be passed to the RF constructor
RF_VALID_PARAMS = ["n_estimators", "max_depth", "min_samples_split", "min_samples_leaf", "max_features", "class_weight"]

def normalize_rf_params(params_like):
    params = {k: v for k, v in dict(params_like).items() if k in RF_VALID_PARAMS}
    # Ensure types are correct for RF
    for int_param in ["n_estimators", "max_depth", "min_samples_split", "min_samples_leaf"]:
        if int_param in params and params[int_param] is not None:
            params[int_param] = int(params[int_param])
    return params

# Storage containers
rf_sbert_best_params_by_subset = {}
rf_sbert_models = {}
rf_sbert_tuning_results = {}
rf_sbert_validation_rows = []

# --- MAIN LOOP ---
for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    print(f"\n===== Random Forest SBERT | subset {subset_name} ({artifact_name}) =====")

    # 1. Load SBERT Validation Split
    validation_data = load_transformer_validation_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    X_train = validation_data["X_train"]
    y_train = np.array(validation_data["y_train"]).ravel()
    X_val = validation_data["X_val"]
    review_ids_val = validation_data["review_ids_val"]
    review_label_map_val = validation_data["review_label_map_val"]

    # 2. Parameter Preparation
    if USE_FIXED_BEST_PARAMS and globals().get('RF_SBERT_FIXED_PARAMS') is not None:
        candidate_param_sets = [normalize_rf_params(RF_SBERT_FIXED_PARAMS)]
    else:
        candidate_param_sets = [
            normalize_rf_params(p) for p in ParameterSampler(
                RF_PARAM_GRID, n_iter=RF_RANDOM_SEARCH_ITER, random_state=RANDOM_STATE
            )
        ]

    # 3. Search / Tuning
    tuning_rows = []
    for candidate_params in tqdm(candidate_param_sets, desc=f"RF SBERT search {subset_name}", leave=False):
        model = RandomForestClassifier(
            **candidate_params,
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        
        # Use your custom smoothing logic for SBERT review-level evaluation
        y_scores, y_pred, y_true = predict_per_review_topk_smooth(
            model.predict_proba(X_val)[:, 1],
            review_ids_val,
            review_label_map_val,
        )
        
        tuning_rows.append({
            **candidate_params,
            "auc": roc_auc_score(y_true, y_scores),
            "average_precision": average_precision_score(y_true, y_scores),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "accuracy": accuracy_score(y_true, y_pred),
        })
        gc.collect()

    # 4. Results Processing
    tuning_df = pd.DataFrame(tuning_rows).sort_values("auc", ascending=False).reset_index(drop=True)
    best_params = normalize_rf_params(tuning_df.iloc[0])
    
    rf_sbert_tuning_results[subset_name] = tuning_df
    rf_sbert_best_params_by_subset[subset_name] = best_params

    print(f"Selected RF parameters for subset {subset_name}: {best_params}")
    display(tuning_df.rename(columns={"average_precision": "pr-auc"}).head(5))

    selected_row = tuning_df.loc[0, list(best_params.keys()) + metric_cols].to_dict()
    rf_sbert_validation_rows.append({"subset": subset_name, **selected_row})

    # 5. Final Training on Full Dataset
    fulltrain_data = load_transformer_fulltrain_test_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    X_train_full = fulltrain_data["X_train_full"]
    y_train_full = np.array(fulltrain_data["y_train_full"]).ravel()

    final_model = RandomForestClassifier(
        **best_params,
        random_state=RANDOM_STATE,
        class_weight=CLASS_WEIGHT,
        n_jobs=-1
    )
    final_model.fit(X_train_full, y_train_full)
    rf_sbert_models[subset_name] = final_model
    
    del validation_data, fulltrain_data
    gc.collect()

# --- FINAL SUMMARY ---
rf_sbert_validation_metrics_df = pd.DataFrame(rf_sbert_validation_rows)
print("\nFinal RF SBERT Validation Summary:")
display(rf_sbert_validation_metrics_df.round(6))

# Assign final references
rf_sbert_best_params = rf_sbert_best_params_by_subset["1"]
rf_sbert_model = rf_sbert_models["1"]

#### 4.7.4 Validation Summary

In [ ]:
RF_METRIC_COLUMNS = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]

rf_validation_summary_sbert = rf_sbert_validation_metrics_df[RF_METRIC_COLUMNS].mean().to_frame().T

validation_summaries = {
    "TF-IDF": validation_summary_rf_tfidf,
    "SBERT": rf_validation_summary_sbert,
}

rf_train_metrics_df = pd.DataFrame(
    [
        {"feature": feature_name, **summary_df.iloc[0].to_dict()}
        for feature_name, summary_df in validation_summaries.items()
    ]
).rename(columns={"average_precision": "pr-auc"})

print("Random Forest validation metrics:")
display(
    rf_train_metrics_df[["feature", "auc", "pr-auc", "f1", "f2", "precision", "recall", "accuracy"]]
    .round(6)
)

### 4.8 XGBoost

#### 4.8.1 TF-IDF

In [ ]:
from xgboost import XGBClassifier

SUBSET_TO_ARTIFACT = {
    "1": "artifacts_1",
    "2": "artifacts_2",
}

metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
xgb_param_casts = {
    "n_estimators": int,
    "max_depth": int,
    "learning_rate": float,
    "subsample": float,
    "colsample_bytree": float,
    "min_child_weight": float,
}

xgb_tfidf_best_params_by_subset = {}
xgb_tfidf_models = {}
xgb_tfidf_tuning_results = {}
xgb_tfidf_validation_rows = []

for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    print(f"\n===== XGBoost TF-IDF | subset {subset_name} ({artifact_name}) =====")

    validation_artifacts = load_validation_split(artifact_name=artifact_name)
    X_train = validation_artifacts["X_tfidf_train"]
    X_val = validation_artifacts["X_tfidf_val"]
    y_train = validation_artifacts["y_train"]
    y_val = validation_artifacts["y_val"]

    if X_train.shape[0] != len(y_train) or X_val.shape[0] != len(y_val):
        raise ValueError(
            f"TF-IDF artifact mismatch for {artifact_name}: "
            f"train={X_train.shape[0]}/{len(y_train)}, val={X_val.shape[0]}/{len(y_val)}"
        )

    train_positive_count = int(np.sum(y_train == 1))
    train_negative_count = int(np.sum(y_train == 0))
    train_scale_pos_weight = train_negative_count / max(train_positive_count, 1)

    artifact_idx = int(subset_name) - 1
    fixed_params = XGB_TFIDF_FIXED_PARAMS[artifact_idx] if artifact_idx < len(XGB_TFIDF_FIXED_PARAMS) else None

    if USE_FIXED_BEST_PARAMS and fixed_params is not None:
        candidate_param_sets = [
            {name: caster(fixed_params[name]) for name, caster in xgb_param_casts.items()}
        ]
        print(f"Using fixed XGBoost parameters for subset {subset_name}: {candidate_param_sets[0]}")
    else:
        candidate_param_sets = [
            {name: caster(params[name]) for name, caster in xgb_param_casts.items()}
            for params in ParameterSampler(
                XGB_PARAM_GRID,
                n_iter=XGB_RANDOM_SEARCH_ITER,
                random_state=RANDOM_STATE,
            )
        ]

    tuning_rows = []
    for candidate_params in tqdm(candidate_param_sets, desc=f"subset {subset_name} XGBoost search", leave=False):
        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
            verbosity=0,
            scale_pos_weight=train_scale_pos_weight,
            **candidate_params,
        )
        model.fit(X_train, y_train)

        y_val_prob = model.predict_proba(X_val)[:, 1]
        tuning_rows.append(
            {
                **candidate_params,
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            }
        )

    tuning_df = pd.DataFrame(tuning_rows).sort_values("auc", ascending=False).reset_index(drop=True)
    best_params = {
        name: caster(tuning_df.loc[0, name])
        for name, caster in xgb_param_casts.items()
    }

    xgb_tfidf_tuning_results[subset_name] = tuning_df
    xgb_tfidf_best_params_by_subset[subset_name] = best_params

    print(f"Selected XGBoost parameters for subset {subset_name}: {best_params}")
    display(
        tuning_df[list(xgb_param_casts) + metric_cols]
        .rename(columns={"average_precision": "pr-auc"})
        .round(6)
    )

    selected_row = tuning_df.loc[0, list(xgb_param_casts) + metric_cols].to_dict()
    xgb_tfidf_validation_rows.append({"subset": subset_name, **selected_row})

    full_artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    X_train_full = full_artifacts["X_tfidf_train_full"]
    y_train_full = full_artifacts["y_train_full"]

    full_positive_count = int(np.sum(y_train_full == 1))
    full_negative_count = int(np.sum(y_train_full == 0))
    full_scale_pos_weight = full_negative_count / max(full_positive_count, 1)

    final_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        verbosity=0,
        scale_pos_weight=full_scale_pos_weight,
        **best_params,
    )
    final_model.fit(X_train_full, y_train_full)
    xgb_tfidf_models[subset_name] = final_model

xgb_tfidf_validation_metrics_df = pd.DataFrame(xgb_tfidf_validation_rows)
print("Validation metrics for selected XGBoost parameters by subset:")
display(
    xgb_tfidf_validation_metrics_df[["subset"] + list(xgb_param_casts) + metric_cols]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

validation_details_xgb_tfidf = xgb_tfidf_validation_metrics_df.copy()
validation_summary_xgb_tfidf = xgb_tfidf_validation_metrics_df[metric_cols].mean().to_frame().T
print("Average validation metrics across subsets:")
display(
    validation_summary_xgb_tfidf.rename(columns={"average_precision": "pr-auc"}).round(6)
)

xgb_tfidf_best_params = xgb_tfidf_best_params_by_subset["1"]
xgb_tfidf_model = xgb_tfidf_models["1"]

#### 4.8.2 SBERT

In [ ]:
from xgboost import XGBClassifier

SUBSET_TO_ARTIFACT = {
    "1": "artifacts_1",
    "2": "artifacts_2",
}

metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
xgb_sbert_param_casts = {
    "n_estimators": int,
    "max_depth": int,
    "learning_rate": float,
    "subsample": float,
    "colsample_bytree": float,
    "min_child_weight": float,
}

xgb_sbert_best_params_by_subset = {}
xgb_sbert_models = {}
xgb_sbert_tuning_results = {}
xgb_sbert_validation_rows = []

for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    print(f"\n===== XGBoost SBERT | subset {subset_name} ({artifact_name}) =====")

    validation_data = load_transformer_validation_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    X_sbert_train_sub = validation_data["X_train"]
    y_sbert_train_sub = validation_data["y_train"]
    X_sbert_val = validation_data["X_val"]
    review_ids_val = validation_data["review_ids_val"]
    review_label_map_val = validation_data["review_label_map_val"]

    if X_sbert_train_sub.shape[0] != len(y_sbert_train_sub):
        raise ValueError(
            f"SBERT artifact mismatch for {artifact_name}: train={X_sbert_train_sub.shape[0]}/{len(y_sbert_train_sub)}"
        )

    train_positive_count = int(np.sum(y_sbert_train_sub == 1))
    train_negative_count = int(np.sum(y_sbert_train_sub == 0))
    train_scale_pos_weight = train_negative_count / max(train_positive_count, 1)

    artifact_idx = int(subset_name) - 1
    fixed_params = XGB_SBERT_FIXED_PARAMS[artifact_idx] if artifact_idx < len(XGB_SBERT_FIXED_PARAMS) else None

    if USE_FIXED_BEST_PARAMS and fixed_params is not None:
        candidate_param_sets = [
            {name: caster(fixed_params[name]) for name, caster in xgb_sbert_param_casts.items()}
        ]
        print(f"Using fixed XGBoost parameters for subset {subset_name}: {candidate_param_sets[0]}")
    else:
        candidate_param_sets = [
            {name: caster(params[name]) for name, caster in xgb_sbert_param_casts.items()}
            for params in ParameterSampler(
                XGB_PARAM_GRID,
                n_iter=XGB_RANDOM_SEARCH_ITER,
                random_state=RANDOM_STATE,
            )
        ]

    tuning_rows = []
    for candidate_params in tqdm(candidate_param_sets, desc=f"SBERT subset {subset_name} XGBoost search", leave=False):
        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
            verbosity=0,
            scale_pos_weight=train_scale_pos_weight,
            **candidate_params,
        )
        model.fit(X_sbert_train_sub, y_sbert_train_sub)

        y_scores, y_pred, y_true = predict_per_review_topk_smooth(
            model.predict_proba(X_sbert_val)[:, 1],
            review_ids_val,
            review_label_map_val,
        )
        tuning_rows.append(
            {
                **candidate_params,
                "auc": roc_auc_score(y_true, y_scores),
                "average_precision": average_precision_score(y_true, y_scores),
                "f1": f1_score(y_true, y_pred, zero_division=0),
                "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "accuracy": accuracy_score(y_true, y_pred),
            }
        )

    tuning_df = pd.DataFrame(tuning_rows).sort_values("auc", ascending=False).reset_index(drop=True)
    best_params = {
        name: caster(tuning_df.loc[0, name])
        for name, caster in xgb_sbert_param_casts.items()
    }

    xgb_sbert_tuning_results[subset_name] = tuning_df
    xgb_sbert_best_params_by_subset[subset_name] = best_params

    print(f"Selected XGBoost parameters for subset {subset_name}: {best_params}")
    display(
        tuning_df[list(xgb_sbert_param_casts) + metric_cols]
        .rename(columns={"average_precision": "pr-auc"})
        .round(6)
    )

    selected_row = tuning_df.loc[0, list(xgb_sbert_param_casts) + metric_cols].to_dict()
    xgb_sbert_validation_rows.append({"subset": subset_name, **selected_row})

    fulltrain_data = load_transformer_fulltrain_test_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    X_sbert_train_full = fulltrain_data["X_train_full"]
    y_sbert_train_full = fulltrain_data["y_train_full"]

    full_positive_count = int(np.sum(y_sbert_train_full == 1))
    full_negative_count = int(np.sum(y_sbert_train_full == 0))
    full_scale_pos_weight = full_negative_count / max(full_positive_count, 1)

    final_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        verbosity=0,
        scale_pos_weight=full_scale_pos_weight,
        **best_params,
    )
    final_model.fit(X_sbert_train_full, y_sbert_train_full)
    xgb_sbert_models[subset_name] = final_model

xgb_sbert_validation_metrics_df = pd.DataFrame(xgb_sbert_validation_rows)
print("Validation metrics for selected XGBoost parameters by subset:")
display(
    xgb_sbert_validation_metrics_df[["subset"] + list(xgb_sbert_param_casts) + metric_cols]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

validation_details_xgb_sbert = xgb_sbert_validation_metrics_df.copy()
validation_summary_xgb_sbert = xgb_sbert_validation_metrics_df[metric_cols].mean().to_frame().T
print("Average validation metrics across subsets:")
display(
    validation_summary_xgb_sbert.rename(columns={"average_precision": "pr-auc"}).round(6)
)

xgb_sbert_best_params = xgb_sbert_best_params_by_subset["1"]
xgb_sbert_model = xgb_sbert_models["1"]

#### 4.8.3 Validation Summary

In [ ]:
XGB_METRIC_COLUMNS = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]

validation_summaries = {
    "TF-IDF": validation_summary_xgb_tfidf,
    "SBERT": validation_summary_xgb_sbert,
}

xgb_train_metrics_df = pd.DataFrame(
    [
        {"feature": feature_name, **summary_df.iloc[0].to_dict()}
        for feature_name, summary_df in validation_summaries.items()
    ]
).rename(columns={"average_precision": "pr-auc"})

print("XGBoost validation metrics:")
display(
    xgb_train_metrics_df[["feature", "auc", "pr-auc", "f1", "f2", "precision", "recall", "accuracy"]]
    .round(6)
)

### 4.9 Naive Bayes

#### 4.9.1 TF-IDF

##### 4.9.1.1 Alpha Tuning

In [ ]:
from sklearn.naive_bayes import MultinomialNB, GaussianNB

NB_METRIC_COLUMNS = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]

def summarize_nb_metrics(df):
    return df[NB_METRIC_COLUMNS].mean().to_frame().T

def compute_nb_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob),
        "average_precision": average_precision_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }

def evaluate_nb_tfidf_artifacts(alpha, threshold=0.5):
    rows = []
    for artifact_name in ARTIFACT_DIRS:
        artifacts = load_validation_split(artifact_name=artifact_name)
        model = MultinomialNB(alpha=alpha)
        model.fit(artifacts["X_tfidf_train"], artifacts["y_train"] )
        y_val_prob = model.predict_proba(artifacts["X_tfidf_val"])[:, 1]
        rows.append({"artifact_dir": artifact_name, **compute_nb_metrics(artifacts["y_val"], y_val_prob, threshold=threshold)})
    return pd.DataFrame(rows)

def evaluate_nb_sbert_artifacts(var_smoothing, threshold=0.5):
    rows = []
    for artifact_name in ARTIFACT_DIRS:
        artifacts = load_sbert_artifact_validation_splits(artifact_name=artifact_name)
        model = GaussianNB(var_smoothing=var_smoothing)
        model.fit(artifacts["X_sbert_train"], artifacts["y_sbert_train"] )
        y_val_prob_window = model.predict_proba(artifacts["X_sbert_val"])[:, 1]
        y_val_true_review, y_val_prob_review = aggregate_review_probs_topk(
            y_val_prob_window,
            artifacts["review_ids_val"],
            artifacts["review_label_map_val"],
            k=NB_SBERT_TOPK,
        )
        rows.append({"artifact_dir": artifact_name, **compute_nb_metrics(y_val_true_review, y_val_prob_review, threshold=threshold)})
    return pd.DataFrame(rows)

if USE_FIXED_BEST_PARAMS and NB_TFIDF_FIXED_ALPHA is not None:
    best_alpha_nb = float(NB_TFIDF_FIXED_ALPHA)
    artifact_validation_metrics_nb_fixed_alpha = evaluate_nb_tfidf_artifacts(best_alpha_nb)
    artifact_validation_summary_nb_fixed_alpha_by_artifact = artifact_validation_metrics_nb_fixed_alpha.copy()
    artifact_validation_summary_nb_fixed_alpha = summarize_nb_metrics(artifact_validation_metrics_nb_fixed_alpha)

    print(f"Using fixed alpha: {best_alpha_nb} (full search skipped)")
    print("Validation metrics by artifact set:")
    display(artifact_validation_metrics_nb_fixed_alpha.rename(columns={"average_precision": "pr-auc"}))
    print("Average metrics across artifact sets:")
    display(artifact_validation_summary_nb_fixed_alpha.rename(columns={"average_precision": "pr-auc"}))
    artifact_validation_summary_nb_selected_alpha = artifact_validation_summary_nb_fixed_alpha

else:
    artifact_validation_search_rows_nb_alpha = []
    artifact_validation_summary_nb_by_alpha_and_artifact = {}

    for alpha in tqdm(NB_ALPHA_GRID, desc="NB TF-IDF alpha search"):
        per_artifact_df = evaluate_nb_tfidf_artifacts(alpha).assign(alpha=alpha)
        artifact_validation_summary_nb_by_alpha_and_artifact[alpha] = per_artifact_df
        overall_summary = summarize_nb_metrics(per_artifact_df).iloc[0]

        artifact_validation_search_rows_nb_alpha.append(
            {
                "alpha": alpha,
                "mean_auc": overall_summary["auc"],
                "mean_average_precision": overall_summary["average_precision"],
                "mean_f1": overall_summary["f1"],
                "mean_f2": overall_summary["f2"],
                "mean_precision": overall_summary["precision"],
                "mean_recall": overall_summary["recall"],
                "mean_accuracy": overall_summary["accuracy"],
            }
        )

    artifact_validation_results_nb_alpha = (
        pd.DataFrame(artifact_validation_search_rows_nb_alpha)
        .sort_values("mean_auc", ascending=False)
        .reset_index(drop=True)
    )

    print("Validation summary across alpha values (ranked by mean AUC across artifact sets):")
    display(artifact_validation_results_nb_alpha.rename(columns={"mean_average_precision": "mean_pr-auc"}))

    best_alpha_nb = float(artifact_validation_results_nb_alpha.loc[0, "alpha"] )
    print(f"\nBest alpha by validation mean AUC: {best_alpha_nb}")

    print("Validation metrics by artifact set for best alpha:")
    display(
        artifact_validation_summary_nb_by_alpha_and_artifact[best_alpha_nb]
        .rename(columns={"average_precision": "pr-auc"})
    )

    artifact_validation_summary_nb_best_alpha = summarize_nb_metrics(artifact_validation_summary_nb_by_alpha_and_artifact[best_alpha_nb])
    print("Average validation metrics across artifact sets for best alpha:")
    display(artifact_validation_summary_nb_best_alpha.rename(columns={"average_precision": "pr-auc"}))
    artifact_validation_summary_nb_selected_alpha = artifact_validation_summary_nb_best_alpha

nb_tfidf_models = {}

for artifact_name in ARTIFACT_DIRS:
    artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    X_train_full = artifacts["X_tfidf_train_full"]
    y_train_full = artifacts["y_train_full"]

    final_model_nb = MultinomialNB(alpha=best_alpha_nb)
    final_model_nb.fit(X_train_full, y_train_full)

    nb_tfidf_models[artifact_name] = final_model_nb

# Keep model(s) and selected hyperparameter available for model evaluation (section 5)
nb_tfidf_best_alpha = best_alpha_nb
print(f"Stored Naive Bayes TF-IDF models for: {list(nb_tfidf_models.keys())}")
print(f"Selected alpha: {nb_tfidf_best_alpha}")

##### 4.9.1.2 Threshold Tuning

In [ ]:
if "best_alpha_nb" not in globals():
    raise ValueError("Run the Naive Bayes alpha-tuning cell first so best_alpha_nb is available.")

if USE_FIXED_BEST_PARAMS and NB_TFIDF_FIXED_THRESHOLD is not None:
    best_threshold_nb = float(NB_TFIDF_FIXED_THRESHOLD)
    artifact_validation_metrics_nb_fixed_threshold = evaluate_nb_tfidf_artifacts(best_alpha_nb, threshold=best_threshold_nb)
    artifact_validation_summary_nb_fixed_threshold_by_artifact = artifact_validation_metrics_nb_fixed_threshold.copy()
    artifact_validation_summary_nb_fixed_threshold = summarize_nb_metrics(artifact_validation_metrics_nb_fixed_threshold)

    print(f"Using fixed threshold: {best_threshold_nb} (full search skipped)")
    print("Validation metrics by artifact set:")
    display(artifact_validation_metrics_nb_fixed_threshold.rename(columns={"average_precision": "pr-auc"}))
    print("Average metrics across artifact sets:")
    display(artifact_validation_summary_nb_fixed_threshold.rename(columns={"average_precision": "pr-auc"}))
    artifact_validation_summary_nb_selected_threshold = artifact_validation_summary_nb_fixed_threshold

else:
    artifact_validation_search_rows_nb_threshold = []
    artifact_validation_summary_nb_by_threshold_and_artifact = {}

    for threshold in tqdm(NB_THRESHOLD_GRID, desc="NB TF-IDF threshold search"):
        per_artifact_df = evaluate_nb_tfidf_artifacts(best_alpha_nb, threshold=threshold).assign(threshold=threshold)
        artifact_validation_summary_nb_by_threshold_and_artifact[threshold] = per_artifact_df
        overall_summary = summarize_nb_metrics(per_artifact_df).iloc[0]

        artifact_validation_search_rows_nb_threshold.append(
            {
                "threshold": threshold,
                "mean_auc": overall_summary["auc"],
                "std_auc": per_artifact_df["auc"].std(),
                "mean_average_precision": overall_summary["average_precision"],
                "mean_f1": overall_summary["f1"],
                "mean_f2": overall_summary["f2"],
                "mean_precision": overall_summary["precision"],
                "mean_recall": overall_summary["recall"],
                "mean_accuracy": overall_summary["accuracy"],
            }
        )

    artifact_validation_results_nb_threshold = (
        pd.DataFrame(artifact_validation_search_rows_nb_threshold)
        .sort_values("mean_f2", ascending=False)
        .reset_index(drop=True)
    )

    print("Validation summary across threshold values (ranked by mean F2 across artifact sets):")
    display(artifact_validation_results_nb_threshold.rename(columns={"mean_average_precision": "mean_pr-auc"}))

    best_threshold_nb = float(artifact_validation_results_nb_threshold.loc[0, "threshold"])
    print(f"\nBest threshold by validation mean F2: {best_threshold_nb}")

    print("Validation metrics by artifact set for best threshold:")
    display(
        artifact_validation_summary_nb_by_threshold_and_artifact[best_threshold_nb]
        .rename(columns={"average_precision": "pr-auc"})
    )

    artifact_validation_summary_nb_best_threshold = summarize_nb_metrics(artifact_validation_summary_nb_by_threshold_and_artifact[best_threshold_nb])
    print("Average validation metrics across artifact sets for best threshold:")
    display(artifact_validation_summary_nb_best_threshold.rename(columns={"average_precision": "pr-auc"}))
    artifact_validation_summary_nb_selected_threshold = artifact_validation_summary_nb_best_threshold

nb_tfidf_best_threshold = best_threshold_nb
print(f"Selected Naive Bayes TF-IDF threshold: {nb_tfidf_best_threshold}")

#### 4.9.2 SBERT Windows

##### 4.9.2.1 Variance Smoothing Tuning

In [ ]:
if USE_FIXED_BEST_PARAMS and NB_SBERT_FIXED_VAR_SMOOTHING is not None:
    best_var_smoothing_nb_sbert = float(NB_SBERT_FIXED_VAR_SMOOTHING)
    artifact_validation_metrics_nb_sbert_fixed_var_smoothing = evaluate_nb_sbert_artifacts(best_var_smoothing_nb_sbert)
    artifact_validation_summary_nb_sbert_fixed_var_smoothing_by_artifact = artifact_validation_metrics_nb_sbert_fixed_var_smoothing.copy()
    artifact_validation_summary_nb_sbert_fixed_var_smoothing = summarize_nb_metrics(artifact_validation_metrics_nb_sbert_fixed_var_smoothing)

    print(f"Using fixed var_smoothing: {best_var_smoothing_nb_sbert:.0e} (full search skipped)")
    print("Validation metrics by artifact set:")
    display(artifact_validation_metrics_nb_sbert_fixed_var_smoothing.rename(columns={"average_precision": "pr-auc"}))
    print("Average validation metrics across artifact sets:")
    display(artifact_validation_summary_nb_sbert_fixed_var_smoothing.rename(columns={"average_precision": "pr-auc"}))
    artifact_validation_summary_nb_sbert_selected_var_smoothing = artifact_validation_summary_nb_sbert_fixed_var_smoothing

else:
    artifact_validation_search_rows_nb_sbert_var_smoothing = []
    artifact_validation_summary_nb_sbert_by_var_smoothing_and_artifact = {}

    for var_smoothing in tqdm(NB_SBERT_VAR_SMOOTHING_GRID, desc="NB SBERT var_smoothing search"):
        per_artifact_df = evaluate_nb_sbert_artifacts(var_smoothing).assign(var_smoothing=var_smoothing)
        artifact_validation_summary_nb_sbert_by_var_smoothing_and_artifact[var_smoothing] = per_artifact_df
        overall_summary = summarize_nb_metrics(per_artifact_df).iloc[0]

        artifact_validation_search_rows_nb_sbert_var_smoothing.append(
            {
                "var_smoothing": var_smoothing,
                "mean_auc": overall_summary["auc"],
                "mean_average_precision": overall_summary["average_precision"],
                "mean_f1": overall_summary["f1"],
                "mean_f2": overall_summary["f2"],
                "mean_precision": overall_summary["precision"],
                "mean_recall": overall_summary["recall"],
                "mean_accuracy": overall_summary["accuracy"],
            }
        )

    artifact_validation_results_nb_sbert_var_smoothing = (
        pd.DataFrame(artifact_validation_search_rows_nb_sbert_var_smoothing)
        .sort_values("mean_auc", ascending=False)
        .reset_index(drop=True)
    )

    print("Validation summary across var_smoothing values (ranked by mean AUC across artifact sets):")
    display(artifact_validation_results_nb_sbert_var_smoothing.rename(columns={"mean_average_precision": "mean_pr-auc"}))

    best_var_smoothing_nb_sbert = float(artifact_validation_results_nb_sbert_var_smoothing.loc[0, "var_smoothing"])
    print(f"\nBest var_smoothing by validation mean AUC: {best_var_smoothing_nb_sbert:.0e}")

    print("Validation metrics by artifact set for best var_smoothing:")
    display(
        artifact_validation_summary_nb_sbert_by_var_smoothing_and_artifact[best_var_smoothing_nb_sbert]
        .rename(columns={"average_precision": "pr-auc"})
    )

    artifact_validation_summary_nb_sbert_best_var_smoothing = summarize_nb_metrics(artifact_validation_summary_nb_sbert_by_var_smoothing_and_artifact[best_var_smoothing_nb_sbert])
    print("Average validation metrics across artifact sets for best var_smoothing:")
    display(artifact_validation_summary_nb_sbert_best_var_smoothing.rename(columns={"average_precision": "pr-auc"}))
    artifact_validation_summary_nb_sbert_selected_var_smoothing = artifact_validation_summary_nb_sbert_best_var_smoothing

nb_sbert_models = {}

for artifact_name in ARTIFACT_DIRS:
    artifacts = load_sbert_artifact_fulltrain_test_splits(artifact_name=artifact_name)
    X_train_full = artifacts["X_sbert_train_full"]
    y_train_full = artifacts["y_sbert_train_full"]

    final_model_nb_sbert = GaussianNB(var_smoothing=best_var_smoothing_nb_sbert)
    final_model_nb_sbert.fit(X_train_full, y_train_full)

    nb_sbert_models[artifact_name] = final_model_nb_sbert

nb_sbert_best_var_smoothing = best_var_smoothing_nb_sbert
print(f"Stored GaussianNB SBERT window models for: {list(nb_sbert_models.keys())}")
print(f"Selected var_smoothing: {nb_sbert_best_var_smoothing:.0e}")

##### 4.9.2.2 Threshold Tuning

In [ ]:
if "best_var_smoothing_nb_sbert" not in globals():
    raise ValueError(
        "Run the SBERT variance-smoothing tuning cell first so best_var_smoothing_nb_sbert is available."
    )

if USE_FIXED_BEST_PARAMS and NB_SBERT_FIXED_THRESHOLD is not None:
    best_threshold_nb_sbert = float(NB_SBERT_FIXED_THRESHOLD)
    artifact_validation_metrics_nb_sbert_fixed_threshold = evaluate_nb_sbert_artifacts(best_var_smoothing_nb_sbert, threshold=best_threshold_nb_sbert)
    artifact_validation_summary_nb_sbert_fixed_threshold_by_artifact = artifact_validation_metrics_nb_sbert_fixed_threshold.copy()
    artifact_validation_summary_nb_sbert_fixed_threshold = summarize_nb_metrics(artifact_validation_metrics_nb_sbert_fixed_threshold)

    print(f"Using fixed threshold: {best_threshold_nb_sbert} (full search skipped)")
    print("Validation metrics by artifact set:")
    display(artifact_validation_metrics_nb_sbert_fixed_threshold.rename(columns={"average_precision": "pr-auc"}))
    print("Average validation metrics across artifact sets:")
    display(artifact_validation_summary_nb_sbert_fixed_threshold.rename(columns={"average_precision": "pr-auc"}))
    artifact_validation_summary_nb_sbert_selected_threshold = artifact_validation_summary_nb_sbert_fixed_threshold

else:
    artifact_validation_search_rows_nb_sbert_threshold = []
    artifact_validation_summary_nb_sbert_by_threshold_and_artifact = {}

    for threshold in tqdm(NB_SBERT_THRESHOLD_GRID, desc="NB SBERT threshold search"):
        per_artifact_df = evaluate_nb_sbert_artifacts(best_var_smoothing_nb_sbert, threshold=threshold).assign(threshold=threshold)
        artifact_validation_summary_nb_sbert_by_threshold_and_artifact[threshold] = per_artifact_df
        overall_summary = summarize_nb_metrics(per_artifact_df).iloc[0]

        artifact_validation_search_rows_nb_sbert_threshold.append(
            {
                "threshold": threshold,
                "mean_auc": overall_summary["auc"],
                "std_auc": per_artifact_df["auc"].std(),
                "mean_average_precision": overall_summary["average_precision"],
                "mean_f1": overall_summary["f1"],
                "mean_f2": overall_summary["f2"],
                "mean_precision": overall_summary["precision"],
                "mean_recall": overall_summary["recall"],
                "mean_accuracy": overall_summary["accuracy"],
            }
        )

    artifact_validation_results_nb_sbert_threshold = (
        pd.DataFrame(artifact_validation_search_rows_nb_sbert_threshold)
        .sort_values("mean_f2", ascending=False)
        .reset_index(drop=True)
    )

    print("Validation summary across threshold values (ranked by mean F2 across artifact sets):")
    display(artifact_validation_results_nb_sbert_threshold.rename(columns={"mean_average_precision": "mean_pr-auc"}))

    best_threshold_nb_sbert = float(artifact_validation_results_nb_sbert_threshold.loc[0, "threshold"])
    print(f"\nBest threshold by validation mean F2: {best_threshold_nb_sbert}")

    print("Validation metrics by artifact set for best threshold:")
    display(
        artifact_validation_summary_nb_sbert_by_threshold_and_artifact[best_threshold_nb_sbert]
        .rename(columns={"average_precision": "pr-auc"})
    )

    artifact_validation_summary_nb_sbert_best_threshold = summarize_nb_metrics(artifact_validation_summary_nb_sbert_by_threshold_and_artifact[best_threshold_nb_sbert])
    print("Average validation metrics across artifact sets for best threshold:")
    display(artifact_validation_summary_nb_sbert_best_threshold.rename(columns={"average_precision": "pr-auc"}))
    artifact_validation_summary_nb_sbert_selected_threshold = artifact_validation_summary_nb_sbert_best_threshold

nb_sbert_best_threshold = best_threshold_nb_sbert
print(f"Selected Naive Bayes SBERT threshold: {nb_sbert_best_threshold}")

#### 4.9.3 Model Selection Summary

In [ ]:
nb_train_metrics_df = pd.DataFrame([
    {"feature": "TF-IDF", **artifact_validation_summary_nb_selected_threshold.iloc[0].to_dict()},
    {"feature": "SBERT Windows", **artifact_validation_summary_nb_sbert_selected_threshold.iloc[0].to_dict()},
]).rename(columns={"average_precision": "pr-auc"})

print("Naive Bayes model-selection metrics:")
display(
    nb_train_metrics_df[["feature", "auc", "pr-auc", "f1", "f2", "precision", "recall", "accuracy"]]
        .round(6)
)

## 5. Final Test Set Evaluation

This section reports held-out test performance after validation-based model selection and final refitting are complete. For each Logistic Regression feature variant, both active artifact streams are evaluated and the final comparison table reports the average held-out metrics across the two streams.

### 5.1 Logistic Regression

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob),
        "average_precision": average_precision_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }

logreg_metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
logreg_test_rows = []

# TF-IDF
for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    y_test = artifacts["y_test"]
    y_test_prob_logreg_tfidf = logreg_tfidf_models[subset_name].predict_proba(artifacts["X_tfidf_test"])[:, 1]
    test_metrics_logreg_tfidf = compute_metrics(y_test, y_test_prob_logreg_tfidf)
    logreg_test_rows.append({"feature": "TF-IDF", "subset": subset_name, **test_metrics_logreg_tfidf})

# SBERT
for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    artifacts = load_transformer_fulltrain_test_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    y_test_prob_logreg_sbert, _, y_test_true_sbert = predict_per_review_topk_smooth(
        logreg_sbert_models[subset_name].predict_proba(artifacts["X_test"])[:, 1],
        artifacts["review_ids_test"],
        artifacts["review_label_map_test"],
    )
    test_metrics_logreg_sbert = compute_metrics(y_test_true_sbert, np.asarray(y_test_prob_logreg_sbert))
    logreg_test_rows.append({"feature": "SBERT", "subset": subset_name, **test_metrics_logreg_sbert})

# RoBERTa
for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    artifacts = load_transformer_fulltrain_test_splits(
        artifact_name=artifact_name,
        model_type="roberta",
    )
    y_test_prob_logreg_roberta, _, y_test_true_roberta = predict_per_review_topk_smooth(
        logreg_roberta_models[subset_name].predict_proba(artifacts["X_test"])[:, 1],
        artifacts["review_ids_test"],
        artifacts["review_label_map_test"],
    )
    test_metrics_logreg_roberta = compute_metrics(y_test_true_roberta, np.asarray(y_test_prob_logreg_roberta))
    logreg_test_rows.append({"feature": "RoBERTa", "subset": subset_name, **test_metrics_logreg_roberta})

logreg_test_metrics_by_subset_df = pd.DataFrame(logreg_test_rows)
print("Held-out test metrics by subset:")
display(
    logreg_test_metrics_by_subset_df[["feature", "subset", "auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

logreg_test_metrics_df = (
    logreg_test_metrics_by_subset_df.groupby("feature", as_index=False)[logreg_metric_cols]
    .mean()
    .rename(columns={"average_precision": "pr-auc"})
)

print("Final held-out test metrics (Logistic Regression):")
display(
    logreg_test_metrics_df[["feature", "auc", "pr-auc", "f1", "f2", "precision", "recall", "accuracy"]]
    .round(6)
)

### 5.2 Support Vector Machine

In [ ]:
svm_metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
svm_test_rows = []

# TF-IDF
for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    y_test = artifacts["y_test"]
    y_test_prob_svm_tfidf = svm_tfidf_models[subset_name].predict_proba(artifacts["X_tfidf_test"])[:, 1]
    test_metrics_svm_tfidf = compute_metrics(y_test, y_test_prob_svm_tfidf)
    svm_test_rows.append({"feature": "TF-IDF", "subset": subset_name, **test_metrics_svm_tfidf})

# SBERT
for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    artifacts = load_transformer_fulltrain_test_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    y_test_prob_svm_sbert, _, y_test_true_sbert = predict_per_review_topk_smooth(
        svm_sbert_models[subset_name].predict_proba(artifacts["X_test"])[:, 1],
        artifacts["review_ids_test"],
        artifacts["review_label_map_test"],
    )
    test_metrics_svm_sbert = compute_metrics(y_test_true_sbert, np.asarray(y_test_prob_svm_sbert))
    svm_test_rows.append({"feature": "SBERT", "subset": subset_name, **test_metrics_svm_sbert})

svm_test_metrics_by_subset_df = pd.DataFrame(svm_test_rows)
print("Held-out test metrics by subset:")
display(
    svm_test_metrics_by_subset_df[["feature", "subset", "auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

test_metrics_svm_df = (
    svm_test_metrics_by_subset_df.groupby("feature", as_index=False)[svm_metric_cols]
    .mean()
    .rename(columns={"average_precision": "pr-auc"})
)

print("\nFinal held-out test metrics (SVM):")
display(
    test_metrics_svm_df[["feature", "auc", "pr-auc", "f1", "f2", "precision", "recall", "accuracy"]]
    .round(6)
)

### 5.3 Decision Tree

In [ ]:
dt_metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
dt_test_rows = []

# TF-IDF
for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    y_test = artifacts["y_test"]
    y_test_prob_dt_tfidf = dt_tfidf_models[subset_name].predict_proba(artifacts["X_tfidf_test"])[:, 1]
    test_metrics_dt_tfidf = compute_metrics(y_test, y_test_prob_dt_tfidf)
    dt_test_rows.append({"feature": "TF-IDF", "subset": subset_name, **test_metrics_dt_tfidf})

# SBERT
for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    artifacts = load_transformer_fulltrain_test_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    y_test_prob_dt_sbert, _, y_test_true_sbert = predict_per_review_topk_smooth(
        dt_sbert_models[subset_name].predict_proba(artifacts["X_test"])[:, 1],
        artifacts["review_ids_test"],
        artifacts["review_label_map_test"],
    )
    test_metrics_dt_sbert = compute_metrics(y_test_true_sbert, np.asarray(y_test_prob_dt_sbert))
    dt_test_rows.append({"feature": "SBERT", "subset": subset_name, **test_metrics_dt_sbert})

dt_test_metrics_by_subset_df = pd.DataFrame(dt_test_rows)
print("Held-out test metrics by subset:")
display(
    dt_test_metrics_by_subset_df[["feature", "subset", "auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

dt_test_metrics_df = (
    dt_test_metrics_by_subset_df.groupby("feature", as_index=False)[dt_metric_cols]
    .mean()
    .rename(columns={"average_precision": "pr-auc"})
)

print("Final held-out test metrics (Decision Tree):")
display(
    dt_test_metrics_df[["feature", "auc", "pr-auc", "f1", "f2", "precision", "recall", "accuracy"]]
    .round(6)
)

### 5.4 Random Forest

In [ ]:
rf_metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
rf_test_rows = []

# TF-IDF
for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    y_test = artifacts["y_test"]
    y_test_prob_rf_tfidf = rf_tfidf_models[subset_name].predict_proba(artifacts["X_tfidf_test"])[:, 1]
    test_metrics_rf_tfidf = compute_metrics(y_test, y_test_prob_rf_tfidf)
    rf_test_rows.append({"feature": "TF-IDF", "subset": subset_name, **test_metrics_rf_tfidf})

# SBERT
for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    artifacts = load_transformer_fulltrain_test_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    y_test_prob_rf_sbert, _, y_test_true_sbert = predict_per_review_topk_smooth(
        rf_sbert_models[subset_name].predict_proba(artifacts["X_test"])[:, 1],
        artifacts["review_ids_test"],
        artifacts["review_label_map_test"],
    )
    test_metrics_rf_sbert = compute_metrics(y_test_true_sbert, np.asarray(y_test_prob_rf_sbert))
    rf_test_rows.append({"feature": "SBERT", "subset": subset_name, **test_metrics_rf_sbert})

rf_test_metrics_by_subset_df = pd.DataFrame(rf_test_rows)
print("Held-out test metrics by subset:")
display(
    rf_test_metrics_by_subset_df[["feature", "subset", "auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

rf_test_metrics_df = (
    rf_test_metrics_by_subset_df.groupby("feature", as_index=False)[rf_metric_cols]
    .mean()
    .rename(columns={"average_precision": "pr-auc"})
)

print("Final held-out test metrics (Random Forest):")
display(
    rf_test_metrics_df[["feature", "auc", "pr-auc", "f1", "f2", "precision", "recall", "accuracy"]]
    .round(6)
)

### 5.5 XGBoost

In [ ]:
xgb_metric_cols = ["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]
xgb_test_rows = []

# TF-IDF
for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    y_test = artifacts["y_test"]
    y_test_prob_xgb_tfidf = xgb_tfidf_models[subset_name].predict_proba(artifacts["X_tfidf_test"])[:, 1]
    test_metrics_xgb_tfidf = compute_metrics(y_test, y_test_prob_xgb_tfidf)
    xgb_test_rows.append({"feature": "TF-IDF", "subset": subset_name, **test_metrics_xgb_tfidf})

# SBERT
for subset_name, artifact_name in SUBSET_TO_ARTIFACT.items():
    artifacts = load_transformer_fulltrain_test_splits(
        artifact_name=artifact_name,
        model_type="sbert",
    )
    y_test_prob_xgb_sbert, _, y_test_true_sbert = predict_per_review_topk_smooth(
        xgb_sbert_models[subset_name].predict_proba(artifacts["X_test"])[:, 1],
        artifacts["review_ids_test"],
        artifacts["review_label_map_test"],
    )
    test_metrics_xgb_sbert = compute_metrics(y_test_true_sbert, np.asarray(y_test_prob_xgb_sbert))
    xgb_test_rows.append({"feature": "SBERT", "subset": subset_name, **test_metrics_xgb_sbert})

xgb_test_metrics_by_subset_df = pd.DataFrame(xgb_test_rows)
print("Held-out test metrics by subset:")
display(
    xgb_test_metrics_by_subset_df[["feature", "subset", "auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]]
    .rename(columns={"average_precision": "pr-auc"})
    .round(6)
)

xgb_test_metrics_df = (
    xgb_test_metrics_by_subset_df.groupby("feature", as_index=False)[xgb_metric_cols]
    .mean()
    .rename(columns={"average_precision": "pr-auc"})
)

print("Final held-out test metrics (XGBoost):")
display(
    xgb_test_metrics_df[["feature", "auc", "pr-auc", "f1", "f2", "precision", "recall", "accuracy"]]
    .round(6)
)

### 5.6 Naive Bayes

In [ ]:
artifact_test_rows_nb = []

for artifact_name in ARTIFACT_DIRS:
    artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    y_test_prob_nb = nb_tfidf_models[artifact_name].predict_proba(artifacts["X_tfidf_test"])[:, 1]
    artifact_test_rows_nb.append(
        {
            "artifact_dir": artifact_name,
            "feature": "TF-IDF",
            "alpha": nb_tfidf_best_alpha,
            "var_smoothing": np.nan,
            "threshold": nb_tfidf_best_threshold,
            **compute_nb_metrics(artifacts["y_test"], y_test_prob_nb, threshold=nb_tfidf_best_threshold),
        }
    )

for artifact_name in ARTIFACT_DIRS:
    artifacts = load_sbert_artifact_fulltrain_test_splits(artifact_name=artifact_name)
    y_test_prob_window = nb_sbert_models[artifact_name].predict_proba(artifacts["X_sbert_test"])[:, 1]
    y_test_review_true, y_test_review_prob = aggregate_review_probs_topk(
        y_test_prob_window,
        artifacts["review_ids_test"],
        artifacts["review_label_map_test"],
        k=NB_SBERT_TOPK,
    )
    artifact_test_rows_nb.append(
        {
            "artifact_dir": artifact_name,
            "feature": "SBERT Windows",
            "alpha": np.nan,
            "var_smoothing": nb_sbert_best_var_smoothing,
            "threshold": nb_sbert_best_threshold,
            **compute_nb_metrics(y_test_review_true, y_test_review_prob, threshold=nb_sbert_best_threshold),
        }
    )

test_metrics_nb_df = pd.DataFrame(artifact_test_rows_nb)
print("\nFinal held-out test metrics by artifact set (Naive Bayes):")
display(
    test_metrics_nb_df[
        [
            "artifact_dir",
            "feature",
            "alpha",
            "var_smoothing",
            "threshold",
            "auc",
            "average_precision",
            "f1",
            "f2",
            "precision",
            "recall",
            "accuracy",
        ]
    ].rename(columns={"average_precision": "pr-auc"})
)

test_metrics_nb_avg = pd.DataFrame([
    {
        "feature": "TF-IDF",
        "alpha": nb_tfidf_best_alpha,
        "var_smoothing": np.nan,
        "threshold": nb_tfidf_best_threshold,
        **summarize_nb_metrics(test_metrics_nb_df[test_metrics_nb_df["feature"] == "TF-IDF"]).iloc[0].to_dict(),
    },
    {
        "feature": "SBERT Windows",
        "alpha": np.nan,
        "var_smoothing": nb_sbert_best_var_smoothing,
        "threshold": nb_sbert_best_threshold,
        **summarize_nb_metrics(test_metrics_nb_df[test_metrics_nb_df["feature"] == "SBERT Windows"]).iloc[0].to_dict(),
    },
])

print("\nAverage held-out test metrics by feature (Naive Bayes):")
display(
    test_metrics_nb_avg[
        [
            "feature",
            "alpha",
            "var_smoothing",
            "threshold",
            "auc",
            "average_precision",
            "f1",
            "f2",
            "precision",
            "recall",
            "accuracy",
        ]
    ].rename(columns={"average_precision": "pr-auc"})
)

In [ ]:
for artifact_name in ["artifacts_1", "artifacts_2"]:
    validation_artifacts = load_validation_split(artifact_name=artifact_name)
    full_artifacts = load_full_train_test(artifact_name=artifact_name, use_fulltrain=True)
    print(artifact_name)
    print("  y_train:", validation_artifacts["y_train"].shape)
    print("  X_tfidf_train:", validation_artifacts["X_tfidf_train"].shape)
    print("  y_val:", validation_artifacts["y_val"].shape)
    print("  X_tfidf_val:", validation_artifacts["X_tfidf_val"].shape)
    print("  y_train_full:", full_artifacts["y_train_full"].shape)
    print("  X_tfidf_train_full:", full_artifacts["X_tfidf_train_full"].shape)